# Montagem de banco de perguntas, pipeline completo

**De uma lista de links do YouTube a um `banco_estruturado.json` pronto para o simulador.**

Este notebook condensa o que antes eram cinco notebooks e dois scripts:

| antes | etapa aqui |
|---|---|
| `00_local_transcricao_diarizacao_v3` | 2. transcrever · 3. diarizar · 4. alinhar |
| `01_atribuicao_papel` | 5. quem é examinador, quem é candidato |
| `02_banco_perguntas` + `03_recuperar_blocos` | 6. pares pergunta → resposta |
| `estruturar_banco.py` | 7. estruturar · 8. revisar · 9. corrigir |
| `montar_banco_final.py` | 11. seleção (opcional) |

O que ele acrescenta: **etapa 1, o download**. Você põe links num CSV e o pipeline
baixa, extrai o áudio e segue até o JSON final sem intervenção.

## Para que serve

Rodar o mesmo processo em **outros concursos**. Banca, cargo e taxonomia de
matérias são parâmetros, não estão escritos no código: é o que permite sair do
MPMG/Promotor e montar banco de Defensoria, Magistratura, Procuradoria.

## As três regras que atravessam o pipeline

1. **O modelo devolve índice, nunca texto.** Nas etapas 5 e 6 o LLM recebe falas
   numeradas e responde com números. A transcrição jamais é reescrita: se o
   modelo pudesse devolver texto, uma paráfrase da resposta do candidato entraria
   no dataset sem deixar rastro, e seria contaminação irrecuperável.
2. **A resposta gravada é GABARITO, não a fala do candidato.** A transcrição é
   âncora do recorte (o que o examinador queria ouvir); o conteúdo correto vem do
   modelo. Onde o candidato errou, o gabarito traz o certo, e é justamente o
   "se estiver errada precisa corrigir" do pedido.
3. **Toda etapa é cacheada em disco.** Rodar de novo continua de onde parou. Isso
   não é conveniência: a etapa 2 leva horas de GPU e a 7 gasta dinheiro de API.

## Saída

`dados/saida/banco_estruturado.json`, no mesmo schema que o simulador já consome,
mais os arquivos de auditoria. A estrutura é superconjunto de `src/saida/`: tem
todos os campos daquele, mais `auditoria_citacoes` e `origem_verificada` por
fundamento legal.

## 0. Ambiente

Instale o que faltar (nenhum comando precisa de `sudo`):

```bash
pip install yt-dlp imageio-ffmpeg faster-whisper pyannote.audio openai pandas tqdm
```

**Token do Hugging Face** (`HF_TOKEN`): o pyannote é aberto mas exige cadastro.
Aceite as condições em `hf.co/pyannote/segmentation-3.0` **e** em
`hf.co/pyannote/speaker-diarization-3.1`: são duas páginas, não uma.

**Chave da OpenAI** (`OPENAI_API_KEY`): nunca escreva no notebook. Use variável de
ambiente ou um `.env` fora do controle de versão.

In [ ]:
import csv, glob, hashlib, json, os, re, shutil, subprocess, sys, time, wave
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm

BASE = Path.cwd() / "dados"
DIR_AUDIO   = BASE / "audio"
DIR_TRANSCR = BASE / "transcricoes"
DIR_BANCO   = BASE / "banco"
DIR_CACHE   = BASE / "cache"
DIR_SAIDA   = BASE / "saida"
DIR_BATCH   = BASE / "batch"          # jsonl enviados e estado dos lotes
for d in (DIR_AUDIO, DIR_TRANSCR, DIR_BANCO, DIR_CACHE, DIR_SAIDA, DIR_BATCH):
    d.mkdir(parents=True, exist_ok=True)

CSV_ENTRADA = Path("videos.csv")

# Chaves só por variável de ambiente (repositório público): OPENAI_API_KEY (etapas 6-9) e HF_TOKEN (etapa 4).
OPENAI_API_KEY = ""
HF_TOKEN       = ""

# a variável de ambiente, se existir, tem precedência
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY") or OPENAI_API_KEY
HF_TOKEN       = os.environ.get("HF_TOKEN") or HF_TOKEN

MODELO_LLM        = "gpt-5.6-sol"     # papel, pares, estruturacao, revisao
REASONING_EFFORT  = "high"            # "high" nos modelos que aceitam; None nos demais
MODELO_WHISPER    = "large-v3"
WHISPER_COMPUTE   = "float16"         # "int8" na CPU

MIN_FALANTES, MAX_FALANTES = 2, 15    # teto FOLGADO: 8 forcava fusao de clusters
SILENCIO_MS       = 400               # VAD: 700 colava o "Certo." na fala seguinte
LIMIAR_TURNO_S    = 0.25              # turno mais curto que isso e tremido da diarizacao
LOTE_PAPEL        = 40                # sentencas por chamada na etapa 5
CONTEXTO_PAPEL    = 6                 # sentencas ja decididas enviadas como continuidade
JANELA            = 6                 # itens reescritos por chamada na etapa 7
CONTEXTO          = 3                 # itens anteriores enviados so como contexto
MAX_TOKENS_SAIDA  = 32000             # inclui os tokens de raciocinio, nao so o texto
MAX_REQUISICOES   = 4000              # trava dura por passada
FORCAR            = False             # True ignora o cache e reprocessa tudo

# "batch": Batch API (50% de desconto, até 24 h); "sincrono": uma chamada por vez, para iterar prompts.
MODO_EXECUCAO     = "batch"
INTERVALO_POLL    = 60                # s entre consultas ao estado do lote

VERSAO_PROMPT = "v3"                  # alterar invalida o cache (MODELO_LLM ja entra no hash)

import torch
GPU = torch.cuda.is_available()
if not GPU:
    WHISPER_COMPUTE = "int8"
print(f"GPU: {torch.cuda.get_device_name(0) if GPU else 'nenhuma (Whisper sera lento)'}")
print(f"OPENAI_API_KEY: {'ok' if OPENAI_API_KEY else 'AUSENTE'}")
print(f"HF_TOKEN:       {'ok' if HF_TOKEN else 'AUSENTE'}")
print(f"saida em: {BASE}")

## 1. A lista de vídeos

Um CSV, uma linha por vídeo. Só `url` é obrigatória; o resto é metadado que viaja
até o JSON final e permite dizer, meses depois, de que prova saiu cada questão.

| coluna | o que é | obrigatória |
|---|---|---|
| `url` | link do YouTube | **sim** |
| `banca` | "Ministério Público de Minas Gerais" | não, mas vai no prompt |
| `cargo` | "Promotor de Justiça" | não, mas vai no prompt |
| `edicao` | "LX" | não |
| `ano` | 2024 | não |
| `grupo_tematico` | "IV" | não |
| `colocacao` | colocação do candidato; **calibrador de qualidade, não veredito** | não |

`banca` e `cargo` entram nos prompts das etapas 5 a 9. É o que faz o pipeline
servir a outro concurso sem tocar em código.

In [ ]:
COLUNAS = ["url", "banca", "cargo", "edicao", "ano", "grupo_tematico", "colocacao", "observacao"]

# UF: vem da observação ("UF=MG") ou, na falta dela, do nome da banca.
ESTADOS = {
    "acre":"AC","alagoas":"AL","amapa":"AP","amazonas":"AM","bahia":"BA","ceara":"CE",
    "distrito federal":"DF","espirito santo":"ES","goias":"GO","maranhao":"MA",
    "mato grosso do sul":"MS","mato grosso":"MT","minas gerais":"MG","para":"PA",
    "paraiba":"PB","parana":"PR","pernambuco":"PE","piaui":"PI",
    "rio de janeiro":"RJ","rio grande do norte":"RN","rio grande do sul":"RS",
    "rondonia":"RO","roraima":"RR","santa catarina":"SC","sao paulo":"SP",
    "sergipe":"SE","tocantins":"TO",
}


def uf_do_registro(r: dict) -> str:
    achado = re.search(r"UF=([A-Z]{2})", r.get("observacao") or "")
    if achado:
        return achado.group(1)
    texto = (r.get("banca") or "").lower()
    for a, b in (("á","a"),("â","a"),("ã","a"),("é","e"),("ê","e"),("í","i"),
                 ("ó","o"),("ô","o"),("õ","o"),("ú","u"),("ç","c")):
        texto = texto.replace(a, b)
    # do mais longo para o mais curto: "mato grosso do sul" antes de "mato grosso"
    for nome in sorted(ESTADOS, key=len, reverse=True):
        if nome in texto:
            return ESTADOS[nome]
    return "BR"

if not CSV_ENTRADA.exists():
    with open(CSV_ENTRADA, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=COLUNAS)
        w.writeheader()
        w.writerow({"url": "https://www.youtube.com/watch?v=EXEMPLO",
                    "banca": "Ministério Público de Minas Gerais",
                    "cargo": "Promotor de Justiça", "edicao": "LX", "ano": "2024",
                    "grupo_tematico": "IV", "colocacao": "1", "observacao": ""})
    print(f"criei um {CSV_ENTRADA} de exemplo, preencha e rode esta célula de novo.")

with open(CSV_ENTRADA, encoding="utf-8-sig") as f:
    linhas = [r for r in csv.DictReader(f) if (r.get("url") or "").strip()]

VIDEOS = []
for i, r in enumerate(linhas, 1):
    if "EXEMPLO" in r["url"]:
        continue
    col = (r.get("colocacao") or "").strip()
    VIDEOS.append({
        "video_id": f"video_{i}",
        "url": r["url"].strip(),
        "banca": (r.get("banca") or "").strip() or "banca examinadora",
        "cargo": (r.get("cargo") or "").strip() or "candidato",
        "edicao": (r.get("edicao") or "").strip(),
        "ano": int(r["ano"]) if (r.get("ano") or "").strip().isdigit() else None,
        "grupo_tematico": (r.get("grupo_tematico") or "").strip() or None,
        "colocacao": int(col) if col.isdigit() else None,
        "observacao": (r.get("observacao") or "").strip(),
        "uf": uf_do_registro(r),
    })

assert VIDEOS, f"nenhum vídeo válido em {CSV_ENTRADA}. Preencha a coluna `url`."
print(f"{len(VIDEOS)} vídeo(s) na fila:")
for v in VIDEOS:
    print(f"  {v['video_id']:9} {v['cargo'][:28]:28} {v['edicao'] or '-':6} "
          f"{v['ano'] or '-'}  colocação={v['colocacao'] or '-'}")

## 2. Baixar e extrair o áudio

O vídeo não interessa, só a faixa de voz. Baixamos o melhor áudio disponível e
convertemos para **WAV PCM 16 bits, 16 kHz, mono**, que é o formato que o Whisper
e o pyannote consomem sem reconversão interna.

Baixar só o áudio (`bestaudio`) em vez do vídeo inteiro economiza banda e tempo:
uma arguição de 90 minutos sai em ~80 MB em vez de vários GB.

O `ffmpeg` vem do pacote `imageio-ffmpeg` (binário estático), então não precisa de
`sudo` nem de instalação no sistema.

In [ ]:
def preparar_ffmpeg():
    """Devolve uma pasta contendo um executavel chamado literalmente `ffmpeg`.

    O binario do `imageio-ffmpeg` se chama `ffmpeg-linux-x86_64-v7.0.2`, e tanto o
    yt-dlp quanto o downloader interno dele procuram pelo nome exato `ffmpeg`: um
    deles no PATH, ignorando `ffmpeg_location`. Por isso criamos um link com o nome
    certo E o colocamos no PATH: apontar so a pasta nao basta.
    """
    sistema = shutil.which("ffmpeg")
    if sistema:
        return str(Path(sistema).parent)
    try:
        import imageio_ffmpeg
        real = Path(imageio_ffmpeg.get_ffmpeg_exe())
    except Exception:
        raise SystemExit("ffmpeg nao encontrado. Rode: pip install imageio-ffmpeg")
    shim = BASE / "bin"
    shim.mkdir(parents=True, exist_ok=True)
    link = shim / "ffmpeg"
    if not link.exists():
        try:
            link.symlink_to(real)
        except OSError:            # sistemas sem symlink
            shutil.copy2(real, link)
    link.chmod(0o755)
    return str(shim)


FFMPEG_DIR = preparar_ffmpeg()
os.environ["PATH"] = FFMPEG_DIR + os.pathsep + os.environ.get("PATH", "")
print(f"ffmpeg: {FFMPEG_DIR}")


def invalidar_video(vid):
    """Apaga o audio e TODO derivado de um video_id, inclusive seus pedacos.

    O video_id e a posicao no CSV. Trocar o CSV e reaproveitar a pasta `dados/`
    fazia `video_1` apontar para um link novo enquanto o cache ainda guardava o
    audio do link antigo; foi assim que duas transmissoes de dia inteiro do MPMG
    foram processadas com o audio de um trecho de 2 min do PC/SP, e rotuladas
    como promotor de Minas. Derivado de audio errado nao tem conserto: apaga.
    """
    apagados = 0
    for pasta in (DIR_AUDIO, DIR_TRANSCR):
        for f in list(pasta.glob(f"{vid}.*")) + list(pasta.glob(f"{vid}_*")):
            f.unlink(); apagados += 1
    arq = DIR_BANCO / "pares_brutos.json"
    if arq.exists():
        brutos = json.loads(arq.read_text(encoding="utf-8"))
        sujos = [k for k in brutos if k.startswith((f"{vid}_b", f"{vid}_p"))]
        for k in sujos:
            del brutos[k]
        arq.write_text(json.dumps(brutos, ensure_ascii=False), encoding="utf-8")
        apagados += len(sujos)
    est_f = DIR_BATCH / "estado.json"
    if est_f.exists():
        est = json.loads(est_f.read_text(encoding="utf-8"))
        for k in [k for k in est if k.startswith(f"papel_{vid}_")]:
            del est[k]
        est_f.write_text(json.dumps(est, ensure_ascii=False, indent=2), encoding="utf-8")
    return apagados


def _duracao_youtube(url):
    import yt_dlp
    with yt_dlp.YoutubeDL({"quiet": True, "no_warnings": True,
                           "skip_download": True, "socket_timeout": 60}) as y:
        return y.extract_info(url, download=False).get("duration") or 0


def baixar_audio(v):
    """Baixa o audio do YouTube e normaliza para WAV 16 kHz mono 16-bit.

    Cada WAV ganha ao lado um `.url` com o link de origem. Cache so vale se o link
    bater; arquivo antigo sem a marca e conferido pela duracao contra o YouTube.
    """
    destino = DIR_AUDIO / f"{v['video_id']}.wav"
    marca = DIR_AUDIO / f"{v['video_id']}.url"
    if destino.exists() and not FORCAR:
        with wave.open(str(destino)) as w:
            dur = w.getnframes() / w.getframerate()
        if marca.exists():
            valido = marca.read_text().strip() == v["url"]
        else:
            try:
                valido = abs(dur - _duracao_youtube(v["url"])) < 120
            except Exception:
                valido = True      # sem rede nao se apaga dado por duvida
            if valido:
                marca.write_text(v["url"])
        if valido:
            print(f"  {v['video_id']}: ja existe ({dur/60:.1f} min), pulando")
            return destino
        n = invalidar_video(v["video_id"])
        print(f"  {v['video_id']}: CACHE ERA DE OUTRO VIDEO ({dur/60:.0f} min): "
              f"{n} arquivos derivados apagados, baixando o certo", flush=True)

    import yt_dlp
    bruto = DIR_AUDIO / f"{v['video_id']}_bruto.%(ext)s"
    opcoes = {
        "format": "bestaudio/best",
        "outtmpl": str(bruto),
        "quiet": True,
        "no_warnings": True,
        "retries": 20, "fragment_retries": 20, "socket_timeout": 60,
        "ffmpeg_location": FFMPEG_DIR,
        # 16 kHz mono já na saída do ffmpeg, o formato do Whisper e do pyannote
        "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "wav"}],
        "postprocessor_args": ["-ar", "16000", "-ac", "1", "-sample_fmt", "s16"],
    }
    with yt_dlp.YoutubeDL(opcoes) as ydl:
        info = ydl.extract_info(v["url"], download=True)
    v["titulo"] = info.get("title", "")
    v["duracao_youtube_s"] = info.get("duration")

    gerado = next(iter(DIR_AUDIO.glob(f"{v['video_id']}_bruto.wav")), None)
    if gerado is None:
        raise RuntimeError(f"{v['video_id']}: yt-dlp nao produziu WAV. "
                           f"Confira o link e se o video e publico.")
    gerado.replace(destino)
    marca.write_text(v["url"])
    with wave.open(str(destino)) as w:
        dur = w.getnframes() / w.getframerate()
    print(f"  {v['video_id']}: {dur/60:.1f} min  |  {info.get('title','')[:60]}")
    return destino


for v in tqdm(VIDEOS, desc="baixando"):
    v["audio"] = str(baixar_audio(v))

pd_ok = True
try:
    import pandas as pd
    display(pd.DataFrame([{k: v.get(k) for k in
                           ("video_id", "titulo", "edicao", "ano", "colocacao")}
                          for v in VIDEOS]))
except Exception:
    pd_ok = False

## 3. Transcrever

`faster-whisper` com `word_timestamps=True`. O tempo por palavra é o que permite,
na etapa 4, cruzar "o que foi dito" com "quem falou" **palavra a palavra** em vez
de por segmento, e é aí que nasce ou não a contaminação entre falantes.

Quatro parâmetros que precisam ser defensáveis:

- **`beam_size=5`**: mantém 5 hipóteses de frase em paralelo em vez de escolher
  gulosamente. Custa tempo, reduz erro.
- **`condition_on_previous_text=False`**: impede o efeito bola de neve em áudio
  longo, em que um erro contamina todo o resto.
- **`initial_prompt`**: enfileira o vocabulário jurídico e enviesa a decodificação.
  Sem ele "habeas corpus" vira "abeas corpus". Custa zero.
- **`word_timestamps=True`**: a base da etapa 4.

A célula **aborta** se nenhuma palavra vier com timestamp, em vez de gerar arquivos
inúteis e o problema aparecer três etapas depois.

In [ ]:
PROMPT_JURIDICO = (
    "Prova oral de concurso público. Arguição sobre Direito Constitucional, Penal, "
    "Processual Penal, Civil, Processual Civil, Administrativo e Direito Coletivo. "
    "Vocabulário: habeas corpus, súmula vinculante, Supremo Tribunal Federal, "
    "Superior Tribunal de Justiça, Ministério Público, ação civil pública, inquérito "
    "civil, termo de ajustamento de conduta, improbidade administrativa, legítima "
    "defesa, dolo eventual, culpa consciente, trânsito em julgado, coisa julgada, "
    "tutela antecipada, agravo de instrumento, artigo, parágrafo, inciso, caput, "
    "excelência."
)

_whisper = None

def transcrever(v):
    global _whisper
    saida = DIR_TRANSCR / f"{v['video_id']}_bruto.json"
    if saida.exists() and not FORCAR:
        d = json.loads(saida.read_text(encoding="utf-8"))
        n = sum(len(s.get("palavras", [])) for s in d["segmentos"])
        if n == 0:
            raise SystemExit(f"{v['video_id']}: transcricao sem palavras. Apague "
                             f"{saida} e rode de novo.")
        print(f"  {v['video_id']}: cache ({n} palavras)")
        return d

    if _whisper is None:
        from faster_whisper import WhisperModel
        print(f"carregando {MODELO_WHISPER} (~3 GB na primeira vez)...")
        _whisper = WhisperModel(MODELO_WHISPER, device="cuda" if GPU else "cpu",
                                compute_type=WHISPER_COMPUTE)

    segmentos, info = _whisper.transcribe(
        v["audio"], language="pt", beam_size=5, vad_filter=True,
        vad_parameters={"min_silence_duration_ms": SILENCIO_MS},
        condition_on_previous_text=False, initial_prompt=PROMPT_JURIDICO,
        word_timestamps=True,
    )
    d = {"video_id": v["video_id"], "modelo": MODELO_WHISPER, "duracao": info.duration,
         "duracao_fala": getattr(info, "duration_after_vad", None), "segmentos": []}

    barra = tqdm(total=round(info.duration), unit="s",
                 desc=f"  {v['video_id']}", leave=True)
    for s in segmentos:
        d["segmentos"].append({
            "inicio": round(s.start, 2), "fim": round(s.end, 2), "texto": s.text.strip(),
            "avg_logprob": round(s.avg_logprob, 3),
            "no_speech_prob": round(s.no_speech_prob, 3),
            "palavras": [{"inicio": round(w.start, 3), "fim": round(w.end, 3),
                          "palavra": w.word, "prob": round(w.probability, 3)}
                         for w in (s.words or [])],
        })
        barra.update(max(0, round(s.end) - barra.n))
    barra.update(max(0, barra.total - barra.n)); barra.close()

    n_pal = sum(len(s["palavras"]) for s in d["segmentos"])
    if n_pal == 0:
        raise SystemExit(f"{v['video_id']}: nenhuma palavra com timestamp. "
                         "faster-whisper >= 0.9 e necessario.")
    saida.write_text(json.dumps(d, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"  {v['video_id']}: {len(d['segmentos'])} segmentos, {n_pal} palavras")
    return d


brutos = {v["video_id"]: transcrever(v) for v in tqdm(VIDEOS, desc="transcrevendo")}

## 4. Diarizar: quem falou

A diarização analisa o **sinal**, não o texto, então não é influenciada pelo
conteúdo. É por isso que ela é confiável para separar vozes e **não** para dizer
qual voz é do candidato: isso é a etapa 5.

**`MAX_FALANTES = 15` é teto folgado, não restrição.** Numa versão anterior o teto
era 8 e um vídeo detectou exatamente 8: o pyannote foi forçado a espremer as
pessoas em 8 clusters e fundiu o presidente da banca com o candidato. Numa sessão
em que o candidato é arguido por vários grupos, o número real é 12-15.

Lemos o WAV com a biblioteca padrão em vez de deixar o pyannote decodificar: o
`torchcodec` que ele usa quebra a cada reinstalação do `torch`, e o áudio já está
no formato certo desde a etapa 2.

In [ ]:
_pipe = None

def ler_wav(caminho):
    # WAV PCM 16-bit sem depender do torchcodec
    with wave.open(str(caminho), "rb") as w:
        if w.getsampwidth() != 2:
            raise ValueError(f"esperado PCM 16-bit, veio {w.getsampwidth()*8}-bit")
        n_canais, taxa = w.getnchannels(), w.getframerate()
        cru = w.readframes(w.getnframes())
    dados = np.frombuffer(cru, dtype=np.int16).astype(np.float32) / 32768.0
    return torch.from_numpy(dados.reshape(-1, n_canais).T.copy()), taxa


def diarizar(v):
    global _pipe
    saida = DIR_TRANSCR / f"{v['video_id']}_falantes.json"
    if saida.exists() and not FORCAR:
        print(f"  {v['video_id']}: cache")
        return json.loads(saida.read_text(encoding="utf-8"))

    assert HF_TOKEN, ("HF_TOKEN ausente. Aceite as condicoes em "
                      "hf.co/pyannote/segmentation-3.0 E em "
                      "hf.co/pyannote/speaker-diarization-3.1 (sao duas paginas).")
    if _pipe is None:
        from contextlib import contextmanager
        import inspect

        from pyannote.audio import Pipeline

        @contextmanager
        def _torch_load_permissivo():
            """Restaura o `weights_only=False` de `torch.load` durante o carregamento.

            O PyTorch 2.6 inverteu o padrao para `weights_only=True`, e os
            checkpoints do pyannote 3.x trazem objetos serializados que isso recusa
            com `UnpicklingError: Weights only load failed`. O patch vale SO dentro
            deste bloco e so para o checkpoint oficial do pyannote no HuggingFace:
            carregar pickle arbitrario de fonte desconhecida continua sendo risco.
            """
            original = torch.load
            def permissivo(*a, **kw):
                # força (não setdefault): o Lightning passa weights_only=True explicitamente
                kw["weights_only"] = False
                return original(*a, **kw)
            torch.load = permissivo
            try:
                yield
            finally:
                torch.load = original

        chave = ("token" if "token" in
                 inspect.signature(Pipeline.from_pretrained).parameters
                 else "use_auth_token")
        with _torch_load_permissivo():
            _pipe = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1",
                                             **{chave: HF_TOKEN})
        # from_pretrained devolve None em vez de erro quando a autenticacao falha
        if _pipe is None:
            raise RuntimeError("pyannote devolveu None: token invalido ou condicoes "
                               "nao aceitas nas DUAS paginas do modelo.")
        if GPU:
            _pipe.to(torch.device("cuda"))

    onda, taxa = ler_wav(v["audio"])
    bruto = _pipe({"waveform": onda, "sample_rate": taxa},
                  min_speakers=MIN_FALANTES, max_speakers=MAX_FALANTES)
    del onda
    anot = getattr(bruto, "speaker_diarization", bruto)   # 3.x: Annotation; 4.x: wrapper
    trechos = [{"inicio": round(t.start, 2), "fim": round(t.end, 2), "falante": f}
               for t, _, f in anot.itertracks(yield_label=True)]
    saida.write_text(json.dumps(trechos, ensure_ascii=False, indent=1), encoding="utf-8")

    n = len({t["falante"] for t in trechos})
    print(f"  {v['video_id']}: {n} falantes, {len(trechos)} trechos")
    if n >= MAX_FALANTES:
        print(f"  {v['video_id']}: ALERTA: bateu no teto de {MAX_FALANTES}. "
              "Clusters podem ter sido fundidos; suba MAX_FALANTES e rode com FORCAR=True.")
    return trechos


if GPU and _whisper is not None:
    del _whisper; _whisper = None      # libera a VRAM antes do pyannote
    import gc; gc.collect(); torch.cuda.empty_cache()

falantes = {v["video_id"]: diarizar(v) for v in tqdm(VIDEOS, desc="diarizando")}

## 5. Alinhar palavra → falante e montar turnos

O Whisper diz "o que e quando"; o pyannote diz "quem e quando". Cruzar os dois por
**segmento** é o que produz contaminação: o segmento médio tem ~3,5 s e atravessa
troca de falante em ~5% dos casos, colando a pergunta do examinador dentro da
resposta do candidato.

Aqui o cruzamento é por **palavra**: cada token tem início e fim próprios e recebe
o falante de maior sobreposição temporal. A fronteira do turno cai no lugar certo.

**Indexação em baldes de 1 segundo.** São ~13 mil palavras contra ~1000 trechos por
vídeo; busca linear daria 13 milhões de comparações. Os trechos do pyannote podem
se sobrepor, o que quebra busca binária por início; baldes são simples de acertar e
rápidos o bastante.

Dois detalhes que custaram tempo para achar:

- token que **não começa com espaço** é continuação de subpalavra (`"Bem"` +
  `"-vinda,"`) e herda o falante anterior, senão a fronteira parte a palavra ao meio
- a asserção de conservação compara **fluxo de caracteres**, não tokens: a fronteira
  de turno insere um espaço que não existia

In [ ]:
def indexar(trechos):
    balde = defaultdict(list)
    for t in trechos:
        for s in range(int(t["inicio"]), int(t["fim"]) + 1):
            balde[s].append(t)
    return balde


def falante_da_palavra(ini, fim, balde):
    cands, vistos = [], set()
    for s in range(int(ini), int(fim) + 1):
        for t in balde.get(s, []):
            k = (t["inicio"], t["fim"], t["falante"])
            if k not in vistos:
                vistos.add(k); cands.append(t)
    ov = Counter()
    for t in cands:
        d = min(fim, t["fim"]) - max(ini, t["inicio"])
        if d > 0:
            ov[t["falante"]] += d
    if ov:
        return ov.most_common(1)[0][0]
    if cands:   # fora de qualquer trecho: o falante mais proximo no tempo
        return min(cands, key=lambda t: min(abs(t["inicio"] - fim),
                                            abs(ini - t["fim"])))["falante"]
    return None


def alinhar(bruto, trechos):
    balde = indexar(trechos)
    palavras, ultimo = [], None
    for seg in bruto["segmentos"]:
        for w in seg["palavras"]:
            f = falante_da_palavra(w["inicio"], w["fim"], balde) or ultimo
            ultimo = f or ultimo
            palavras.append({**w, "falante": f})
    # continuacao de subpalavra herda o falante anterior (~0,5% dos tokens)
    for i in range(1, len(palavras)):
        if not palavras[i]["palavra"].startswith(" "):
            palavras[i]["falante"] = palavras[i - 1]["falante"]
    primeiro = next((p["falante"] for p in palavras if p["falante"]), "SPEAKER_00")
    for p in palavras:
        if p["falante"] is None:
            p["falante"] = primeiro
        else:
            break
    return palavras


def montar_turnos(palavras):
    def anexar(t, p):
        t["fim"] = p["fim"]
        t["texto"] += p.get("texto", p.get("palavra", ""))
        t["n_palavras"] += p.get("n_palavras", 1)
        t["_probs"] += p.get("_probs", [p.get("prob")])

    turnos = []
    for p in palavras:
        if turnos and turnos[-1]["falante"] == p["falante"]:
            anexar(turnos[-1], p)
        else:
            turnos.append({"inicio": p["inicio"], "fim": p["fim"], "falante": p["falante"],
                           "texto": p["palavra"], "n_palavras": 1, "_probs": [p["prob"]]})
    limpos = []
    for i, t in enumerate(turnos):
        curto = (t["fim"] - t["inicio"]) < LIMIAR_TURNO_S
        cercado = (0 < i < len(turnos) - 1
                   and turnos[i - 1]["falante"] == turnos[i + 1]["falante"])
        if limpos and (curto and cercado or limpos[-1]["falante"] == t["falante"]):
            anexar(limpos[-1], t)
        else:
            limpos.append(dict(t))
    for t in limpos:
        t["texto"] = t["texto"].strip()
        t["inicio"], t["fim"] = round(t["inicio"], 2), round(t["fim"], 2)
        t["prob_media"] = round(sum(t["_probs"]) / len(t["_probs"]), 3)
        del t["_probs"]
    return limpos


sem_espaco = lambda x: re.sub(r"\s+", "", x)
alinhadas, turnados = {}, {}
for v in tqdm(VIDEOS, desc="alinhando"):
    vid = v["video_id"]
    alinhadas[vid] = alinhar(brutos[vid], falantes[vid])
    turnados[vid] = montar_turnos(alinhadas[vid])

    entrada = [p["palavra"] for p in alinhadas[vid]]
    assert sum(t["n_palavras"] for t in turnados[vid]) == len(entrada), f"{vid}: contagem"
    assert sem_espaco("".join(entrada)) == \
           sem_espaco(" ".join(t["texto"] for t in turnados[vid])), f"{vid}: texto alterado"

    (DIR_TRANSCR / f"{vid}_palavras.json").write_text(
        json.dumps(alinhadas[vid], ensure_ascii=False), encoding="utf-8")
    print(f"  {vid}: {len(turnados[vid])} turnos, {len(entrada)} palavras preservadas")

## 6. Quem é examinador, quem é candidato

O pyannote entrega rótulos anônimos (`SPEAKER_03`). Dizer qual é o candidato exige
olhar o **conteúdo**, e a unidade de decisão é a **sentença**, não o turno: o erro
que se quer corrigir é justamente o caso em que examinador e candidato estão dentro
do mesmo turno.

**Regra de projeto: o modelo devolve índice, nunca texto.** Ele recebe sentenças
numeradas e responde `{"i": 47, "papel": "examinador"}`. A transcrição jamais é
reescrita.

**O `falante` entra como sinal fraco.** Vai no payload como pista, com instrução
explícita de que erra na fronteira do turno, que é o erro que queremos corrigir.

O ponto sensível é a **pergunta retórica**: o candidato usa o tempo todo ("E qual
seria a medida cabível? Primeiro..."), e um classificador ingênuo lê interrogação e
conclui examinador. O prompt trata esse caso de forma explícita.

In [ ]:
from openai import OpenAI
assert OPENAI_API_KEY, "defina OPENAI_API_KEY no ambiente."
cliente = OpenAI(api_key=OPENAI_API_KEY)

ABREV = {"art","arts","inc","incs","par","n","nº","no","sr","sra","srs","dr","dra",
         "exmo","exma","prof","profa","ed","p","pp","cf","cp","cpp","cc","cpc",
         "stf","stj","mp","v","vs","etc","ex"}
MAX_PALAVRAS_SENT = 60


def corpo(system, user, max_tokens=16000):
    b = {"model": MODELO_LLM,
         "messages": [{"role": "system", "content": system},
                      {"role": "user", "content": user}],
         "response_format": {"type": "json_object"},
         "max_completion_tokens": max_tokens}
    if REASONING_EFFORT:
        b["reasoning_effort"] = REASONING_EFFORT
    return b


def chamar(system, user, max_tokens=16000, tentativas=4):
    for n in range(tentativas):
        try:
            r = cliente.chat.completions.create(**corpo(system, user, max_tokens))
            if r.choices[0].finish_reason == "length":
                raise ValueError("resposta truncada (finish_reason=length)")
            return json.loads(r.choices[0].message.content)
        except Exception as e:
            # falta de credito nao melhora esperando: aborta com instrucao clara
            if "insufficient_quota" in str(e) or "no credits" in str(e).lower():
                raise RuntimeError("Sem créditos na OpenAI. Recarregue e rode de novo; "
                                   "o cache preserva o que já foi processado.") from None
            if n == tentativas - 1:
                raise
            time.sleep(2 ** n)


# Batch API: metade do preço, janela de 24 h. O estado de cada lote fica em batch/estado.json,
# então uma queda não reenvia nem repaga.

TERMINAIS = {"completed", "failed", "expired", "cancelled"}


def ler_estado():
    f = DIR_BATCH / "estado.json"
    return json.loads(f.read_text(encoding="utf-8")) if f.exists() else {}


def gravar_estado(est):
    (DIR_BATCH / "estado.json").write_text(
        json.dumps(est, ensure_ascii=False, indent=2), encoding="utf-8")


def _valida(conteudo, ids_esperados):
    dados = json.loads(conteudo)
    chave = "papeis" if "papeis" in dados else ("pares" if "pares" in dados else "itens")
    itens = dados.get(chave)
    if not isinstance(itens, list):
        raise ValueError(f"campo '{chave}' ausente ou nao e lista")
    if chave == "itens" and ids_esperados:
        recebidos, esperados = {i.get("id") for i in itens}, set(ids_esperados)
        if recebidos != esperados:
            raise ValueError(f"ids divergentes: faltando {esperados - recebidos}")
    return dados


def _com_rede(fn, *a, tentativas=12, **kw):
    """Repete uma chamada de rede com espera crescente.

    Internet instavel nao deve derrubar um processo de 10 horas: uma queda de tres
    minutos na madrugada custaria a noite inteira. Ate ~30 min de tolerancia.
    """
    for n in range(tentativas):
        try:
            return fn(*a, **kw)
        except Exception as e:
            if n == tentativas - 1:
                raise
            espera = min(300, 5 * 2 ** n)
            print(f"    rede instavel ({type(e).__name__}: {str(e)[:80]}); "
                  f"nova tentativa em {espera}s", flush=True)
            time.sleep(espera)


def pendentes_de(reqs):
    prontos, faltam = {}, []
    for r in reqs:
        f = DIR_CACHE / f"{r['custom_id']}.json"
        if f.exists() and not FORCAR:
            prontos[r["custom_id"]] = json.loads(f.read_text(encoding="utf-8"))
        else:
            faltam.append(r)
    return prontos, faltam


def submeter_lote(reqs, tag, max_tokens=None):
    """Envia o lote e VOLTA NA HORA, sem esperar.

    E o que permite sobrepor GPU e API: assim que um video e alinhado, o lote dele
    ja esta na fila da OpenAI enquanto a placa transcreve o proximo.
    """
    _, pendentes = pendentes_de(reqs)
    if not pendentes:
        return 0
    estado = ler_estado()
    chave = f"{tag}_r1"
    if chave in estado:
        return 0
    caminho = DIR_BATCH / f"{chave}.jsonl"
    with open(caminho, "w", encoding="utf-8") as f:
        for r in pendentes:
            f.write(json.dumps({
                "custom_id": r["custom_id"], "method": "POST",
                "url": "/v1/chat/completions",
                "body": corpo(r["system"], r["user"], max_tokens or MAX_TOKENS_SAIDA),
            }, ensure_ascii=False) + "\n")
    arq = _com_rede(lambda: cliente.files.create(file=open(caminho, "rb"), purpose="batch"))
    lote = _com_rede(lambda: cliente.batches.create(
        input_file_id=arq.id, endpoint="/v1/chat/completions",
        completion_window="24h", metadata={"tag": chave}))
    estado = ler_estado()
    estado[chave] = {"batch_id": lote.id, "n": len(pendentes), "status": lote.status}
    gravar_estado(estado)
    print(f"  [{tag}] despachado: {lote.id}, {len(pendentes)} req (50% off)", flush=True)
    return len(pendentes)


def coletar_lotes(prefixo: str):
    """Espera e recolhe todos os lotes ja despachados cujo tag comeca com `prefixo`.

    Os lotes enviados video a video durante as etapas 1-4 terminam fora de ordem.
    Esta funcao passa por todos, espera cada um chegar a estado terminal e grava as
    respostas no cache, de onde `executar_lote` as le sem reenviar nada. O que
    falhar simplesmente nao entra no cache, e a passada seguinte reenvia so isso.
    """
    estado = ler_estado()
    chaves = sorted(k for k in estado if k.startswith(prefixo))
    if not chaves:
        return 0
    print(f"  coletando {len(chaves)} lote(s) de '{prefixo}'...", flush=True)
    total = 0
    for k in chaves:
        info = estado[k]
        lote = _com_rede(lambda: cliente.batches.retrieve(info["batch_id"]))
        while lote.status not in TERMINAIS:
            c = lote.request_counts
            print(f"    {k}: {lote.status} {getattr(c,'completed',0)}/"
                  f"{getattr(c,'total',info['n'])}", flush=True)
            time.sleep(INTERVALO_POLL)
            lote = _com_rede(lambda: cliente.batches.retrieve(info["batch_id"]))
        info["status"] = lote.status
        estado[k] = info
        gravar_estado(estado)
        if not lote.output_file_id:
            print(f"    {k}: terminou em '{lote.status}' sem saida", flush=True)
            continue
        texto = _com_rede(lambda: cliente.files.content(lote.output_file_id).text)
        n_ok = 0
        for linha in texto.splitlines():
            if not linha.strip():
                continue
            o = json.loads(linha)
            try:
                esc = o["response"]["body"]["choices"][0]
                if esc.get("finish_reason") == "length":
                    continue
                dados = _valida(esc["message"]["content"], None)
                (DIR_CACHE / f"{o['custom_id']}.json").write_text(
                    json.dumps(dados, ensure_ascii=False, indent=1), encoding="utf-8")
                n_ok += 1
            except Exception:
                pass          # fica fora do cache: a passada seguinte reenvia
        total += n_ok
        print(f"    {k}: {n_ok} respostas gravadas", flush=True)
    return total


def executar_batch(reqs, tag, max_tokens=None):
    """Envia `reqs` num lote, espera terminar e devolve ({custom_id: dados}, falhas).

    Ate MAX_RODADAS reenvios do que falhar: um item truncado ou com ids divergentes
    volta para a fila em vez de derrubar a passada inteira.
    """
    resultados, pendentes = {}, []
    for r in reqs:
        cache = DIR_CACHE / f"{r['custom_id']}.json"
        if cache.exists() and not FORCAR:
            resultados[r["custom_id"]] = json.loads(cache.read_text(encoding="utf-8"))
        else:
            pendentes.append(r)
    if not pendentes:
        print(f"  [{tag}] {len(resultados)} em cache, nada a enviar", flush=True)
        return resultados, []
    if len(pendentes) > MAX_REQUISICOES:
        raise RuntimeError(f"{len(pendentes)} requisicoes em '{tag}' excede a trava "
                           f"de {MAX_REQUISICOES}.")

    estado = ler_estado()
    for rodada in range(1, 4):
        if not pendentes:
            break
        chave = f"{tag}_r{rodada}"
        info = estado.get(chave)

        if not info:
            caminho = DIR_BATCH / f"{chave}.jsonl"
            with open(caminho, "w", encoding="utf-8") as f:
                for r in pendentes:
                    f.write(json.dumps({
                        "custom_id": r["custom_id"], "method": "POST",
                        "url": "/v1/chat/completions",
                        "body": corpo(r["system"], r["user"],
                                      max_tokens or MAX_TOKENS_SAIDA),
                    }, ensure_ascii=False) + "\n")
            arq = _com_rede(lambda: cliente.files.create(file=open(caminho, "rb"),
                                                         purpose="batch"))
            lote = _com_rede(lambda: cliente.batches.create(
                input_file_id=arq.id, endpoint="/v1/chat/completions",
                completion_window="24h", metadata={"tag": chave}))
            info = {"batch_id": lote.id, "n": len(pendentes), "status": lote.status}
            estado[chave] = info
            gravar_estado(estado)
            print(f"  [{tag}] lote {lote.id} enviado, {len(pendentes)} requisicoes "
                  f"(50% de desconto)", flush=True)

        lote = _com_rede(lambda: cliente.batches.retrieve(info["batch_id"]))
        while lote.status not in TERMINAIS:
            c = lote.request_counts
            print(f"  [{tag}] {lote.status}: {getattr(c,'completed',0)}/"
                  f"{getattr(c,'total',info['n'])} "
                  f"(falhas {getattr(c,'failed',0)})", flush=True)
            time.sleep(INTERVALO_POLL)
            lote = _com_rede(lambda: cliente.batches.retrieve(info["batch_id"]))
        info["status"] = lote.status
        estado[chave] = info
        gravar_estado(estado)

        if lote.status != "completed" and not lote.output_file_id:
            # reenviar sozinho gastaria de novo sem o usuario pedir
            raise RuntimeError(
                f"lote {lote.id} terminou em '{lote.status}' sem saida. Remova a "
                f"entrada '{chave}' de {DIR_BATCH/'estado.json'} para reenviar.")

        por_id = {}
        if lote.output_file_id:
            saida_txt = _com_rede(lambda: cliente.files.content(lote.output_file_id).text)
            for linha in saida_txt.splitlines():
                if linha.strip():
                    o = json.loads(linha)
                    por_id[o["custom_id"]] = o
        if lote.error_file_id:
            (DIR_BATCH / f"{chave}_erros.jsonl").write_text(
                cliente.files.content(lote.error_file_id).text, encoding="utf-8")

        restantes = []
        for r in pendentes:
            o = por_id.get(r["custom_id"])
            try:
                if o is None:
                    raise ValueError("sem resposta no arquivo de saida")
                if o.get("error"):
                    raise ValueError(str(o["error"])[:200])
                esc = o["response"]["body"]["choices"][0]
                if esc.get("finish_reason") == "length":
                    raise ValueError("resposta truncada (finish_reason=length)")
                dados = _valida(esc["message"]["content"], r.get("ids"))
                (DIR_CACHE / f"{r['custom_id']}.json").write_text(
                    json.dumps(dados, ensure_ascii=False, indent=1), encoding="utf-8")
                resultados[r["custom_id"]] = dados
            except Exception as e:
                r["_erro"] = str(e)[:200]
                restantes.append(r)
        print(f"  [{tag}] rodada {rodada}: {len(pendentes)-len(restantes)} ok, "
              f"{len(restantes)} a reprocessar", flush=True)
        pendentes = restantes

    return resultados, [{"custom_id": r["custom_id"], "erro": r.get("_erro", "?")}
                        for r in pendentes]


def executar_lote(reqs, tag, max_tokens=None):
    """Roteia para a Batch API ou para chamadas sincronas, conforme MODO_EXECUCAO."""
    if MODO_EXECUCAO == "batch":
        return executar_batch(reqs, tag, max_tokens)
    resultados, falhas = {}, []
    pend = [r for r in reqs if not (DIR_CACHE / f"{r['custom_id']}.json").exists() or FORCAR]
    for r in reqs:
        f = DIR_CACHE / f"{r['custom_id']}.json"
        if f.exists() and not FORCAR:
            resultados[r["custom_id"]] = json.loads(f.read_text(encoding="utf-8"))
    print(f"  [{tag}] {len(resultados)} em cache, {len(pend)} a processar", flush=True)
    for r in tqdm(pend, desc=f"  {tag}", leave=False):
        try:
            dados = chamar(r["system"], r["user"], max_tokens or MAX_TOKENS_SAIDA)
            _valida(json.dumps(dados, ensure_ascii=False), r.get("ids"))
            (DIR_CACHE / f"{r['custom_id']}.json").write_text(
                json.dumps(dados, ensure_ascii=False, indent=1), encoding="utf-8")
            resultados[r["custom_id"]] = dados
        except Exception as e:
            falhas.append({"custom_id": r["custom_id"], "erro": str(e)[:200]})
    return resultados, falhas


def hash_curto(obj):
    return hashlib.sha256(json.dumps(obj, ensure_ascii=False, sort_keys=True)
                          .encode()).hexdigest()[:20]


def fim_de_sentenca(token):
    t = token.strip()
    if not t or t[-1] not in ".?!":
        return False
    if t[-1] == ".":
        raiz = re.sub(r"[^\wº]", "", t[:-1]).lower()
        if raiz in ABREV or (len(raiz) <= 1 and not raiz.isdigit()):
            return False
    return True


def segmentar(palavras):
    sents, buf = [], []
    def fechar():
        sents.append({
            "i": len(sents), "inicio": round(buf[0]["inicio"], 2),
            "fim": round(buf[-1]["fim"], 2),
            "falante": Counter(x["falante"] for x in buf).most_common(1)[0][0],
            "texto": "".join(x["palavra"] for x in buf).strip(),
            "n_palavras": len(buf),
            "prob_media": round(sum(x["prob"] for x in buf) / len(buf), 3)})
    for i, w in enumerate(palavras):
        buf.append(w)
        troca = i + 1 < len(palavras) and palavras[i + 1]["falante"] != w["falante"]
        if fim_de_sentenca(w["palavra"]) or troca or len(buf) >= MAX_PALAVRAS_SENT:
            fechar(); buf = []
    if buf:
        fechar()
    return sents


SYSTEM_PAPEL = """Você classifica falas de uma prova oral de concurso público para o cargo de {cargo} ({banca}).

CENÁRIO
- Uma banca com vários examinadores e UM único candidato.
- Os examinadores fazem perguntas, comentam brevemente e passam a palavra entre si.
- O candidato responde, geralmente em falas longas e argumentativas.

MARCAS DE EXAMINADOR
- Pergunta dirigida ao candidato: "o senhor...?", "a senhora...?", "me diga", "indago".
- Avaliação da resposta alheia: "perfeito", "correto", "certo", "muito bem".
- Condução da sessão: "passo a palavra", "próxima pergunta", "encerro minha inquirição", "senhor presidente".
- Correção ou provocação: "tem certeza?", "e se fosse o contrário?".

MARCAS DE CANDIDATO
- Tratamento deferente: "excelência", "peço vênia", "data venia".
- Hedge: "salvo engano", "se não me falha a memória", "acredito que", "s.m.j.".
- Exposição doutrinária longa, com citação de artigo, súmula e posição de tribunal.
- Pedido de esclarecimento: "poderia repetir?".

REGRA CRÍTICA: PERGUNTA RETÓRICA
O candidato faz perguntas a si mesmo para estruturar a resposta:
  "E qual seria a medida cabível? Primeiro, cumpre observar que..."
Isso é CANDIDATO. Interrogação sozinha não indica examinador. O que indica examinador é a pergunta DIRIGIDA a alguém.

SOBRE O CAMPO "falante"
É um rótulo de agrupamento de voz, automático. Acerta na maior parte, mas ERRA na fronteira entre turnos, que é exatamente o que você deve corrigir. Use como pista, não como verdade.

SAÍDA
JSON puro, sem markdown:
{{"papeis": [{{"i": 0, "papel": "examinador"}}, {{"i": 1, "papel": "candidato"}}]}}
Um item para CADA índice recebido em "classificar". Nunca devolva o texto das sentenças.
Valores permitidos: "examinador" ou "candidato"."""


def reqs_papel(v, sents):
    """Monta as requisicoes de um video. Em lote nao ha sequenciamento, entao o
    contexto vai sem o papel ja decidido: o texto anterior e o `falante` continuam
    ancorando a continuidade, que e o que mais pesa."""
    system = SYSTEM_PAPEL.format(cargo=v["cargo"], banca=v["banca"])
    reqs = []
    for ini in range(0, len(sents), LOTE_PAPEL):
        bloco = sents[ini:ini + LOTE_PAPEL]
        ctx = [{"i": s["i"], "falante": s["falante"], "texto": s["texto"][:200]}
               for s in sents[max(0, ini - CONTEXTO_PAPEL):ini]]
        user = json.dumps({"contexto_anterior": ctx,
                           "classificar": [{"i": s["i"], "falante": s["falante"],
                                            "texto": s["texto"]} for s in bloco]},
                          ensure_ascii=False)
        reqs.append({"custom_id": "papel_" + hash_curto([VERSAO_PROMPT, MODELO_LLM,
                                                         v["video_id"], ini,
                                                         [s["texto"] for s in bloco]]),
                     "system": system, "user": user, "ids": None,
                     "_video": v["video_id"], "_ids": [s["i"] for s in bloco]})
    return reqs


def gravar_papeis(v, sents, reqs, res):
    saida = DIR_TRANSCR / f"{v['video_id']}_papeis.json"
    decidido, fallback = {}, 0
    for r in reqs:
        for item in res.get(r["custom_id"], {}).get("papeis", []):
            if item.get("papel") in ("examinador", "candidato"):
                decidido[int(item["i"])] = item["papel"]
    for s in sents:
        if s["i"] not in decidido:
            decidido[s["i"]] = "candidato"
            fallback += 1
    r = {"video_id": v["video_id"], "modelo": MODELO_LLM, "fallback": fallback,
         "papeis": {str(k): x for k, x in sorted(decidido.items())}}
    saida.write_text(json.dumps(r, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"  {v['video_id']}: {len(decidido)} sentencas, {fallback} fallback", flush=True)
    return r


def remontar(sents, papeis):
    turnos = []
    for s in sents:
        p = papeis["papeis"].get(str(s["i"]), "candidato")
        if turnos and turnos[-1]["papel"] == p:
            t = turnos[-1]
            t["fim"] = s["fim"]; t["texto"] += " " + s["texto"]
            t["n_palavras"] += s["n_palavras"]
            t["_f"].append(s["falante"]); t["_p"].append(s["prob_media"])
        else:
            turnos.append({"inicio": s["inicio"], "fim": s["fim"], "papel": p,
                           "texto": s["texto"], "n_palavras": s["n_palavras"],
                           "_f": [s["falante"]], "_p": [s["prob_media"]]})
    for t in turnos:
        t["falante"] = Counter(t["_f"]).most_common(1)[0][0]
        t["n_falantes_no_turno"] = len(set(t["_f"]))
        t["prob_media"] = round(sum(t["_p"]) / len(t["_p"]), 3)
        del t["_f"], t["_p"]
    return turnos


# sentencas de todos os videos, depois UM lote com tudo
sentencas, reqs_todos = {}, []
for v in VIDEOS:
    vid = v["video_id"]
    s = segmentar(alinhadas[vid])
    assert sem_espaco("".join(w["palavra"] for w in alinhadas[vid])) == \
           sem_espaco(" ".join(x["texto"] for x in s)), f"{vid}: texto alterado"
    sentencas[vid] = s
    reqs_todos += reqs_papel(v, s)

print(f"etapa 5: {len(reqs_todos)} requisicoes de papel "
      f"({sum(len(x) for x in sentencas.values())} sentencas)", flush=True)
res_papel, falhas_papel = executar_lote(reqs_todos, "papel", max_tokens=16000)
assert not falhas_papel, f"papel falhou: {falhas_papel[:3]}"

corpus = {}
for v in VIDEOS:
    vid = v["video_id"]
    sents = sentencas[vid]
    papeis = gravar_papeis(v, sents, [r for r in reqs_todos if r["_video"] == vid], res_papel)
    turnos = remontar(sents, papeis)
    meta = {k: v[k] for k in v if k != "audio"}
    corpus[vid] = {**meta, "modelo_papel": MODELO_LLM, "turnos": turnos}
    (DIR_TRANSCR / f"{vid}_final.json").write_text(
        json.dumps(corpus[vid], ensure_ascii=False, indent=1), encoding="utf-8")

    c = [t for t in turnos if t["papel"] == "candidato"]
    tot = sum(t["n_palavras"] for t in turnos) or 1
    print(f"  {vid}: {len(turnos)} turnos | "
          f"{100*sum(t['n_palavras'] for t in c)/tot:.1f}% da fala é do candidato")

## 7. Pares pergunta → resposta

Parear turno com turno seguinte não funciona: ~30% dos turnos do examinador não têm
interrogação (são avaliação, condução ou correção) e ~19% dos turnos do candidato
têm até 3 palavras ("Sim, excelência"): reconhecimento, não resposta. Filtrar por
tamanho também não resolve: *"E mandado de segurança?"* é pergunta legítima de 4
palavras.

A unidade certa é o **bloco de arguição**: a sequência contínua em que um mesmo
examinador conduz. Cabe um bloco inteiro por chamada, com contexto suficiente para o
modelo saber que a pergunta da linha 3 continua na linha 5 depois de um "Sim,
excelência" no meio. Foi esse agrupamento que levou a cobertura de ordem canônica
de 13% para 83%.

A taxonomia de matérias é **fechada** de propósito: texto livre produziria
"Processual Penal", "Direito Processual Penal" e "Proc. Penal" como três categorias.
Ajuste `MATERIAS` para o edital do seu concurso.

`reacao_examinador` é o achado que sustenta a etapa 9: quando o examinador responde
"perfeito" contra "não é bem assim, veja...", ele está rotulando a resposta de graça.
É a única supervisão humana real do corpus.

In [ ]:
# Taxonomia fechada e por concurso: texto livre criaria categorias duplicadas, e matéria de
# outro edital levaria o modelo a forçar a classificação.
TAXONOMIAS = {
    "mp": [
        "Direito Constitucional", "Direito Penal", "Direito Processual Penal",
        "Direito Civil", "Direito Processual Civil", "Direito Administrativo",
        "Direitos Difusos e Coletivos", "Direito da Infância e Juventude",
        "Direito Eleitoral", "Direito Empresarial", "Direito Tributário",
        "Direito Ambiental", "Direito do Consumidor", "Direitos Humanos",
        "Direito Institucional do Ministério Público", "Outro",
    ],
    "delegado": [
        "Direito Penal", "Direito Processual Penal", "Legislação Penal Especial",
        "Direito Constitucional", "Direito Administrativo", "Direito Civil",
        "Direito Processual Civil", "Criminologia", "Medicina Legal",
        "Direitos Humanos", "Direito Ambiental", "Direito Eleitoral",
        "Direito da Infância e Juventude", "Direito do Consumidor", "Outro",
    ],
}

TAXONOMIAS["magistratura"] = [
    "Direito Constitucional", "Direito Civil", "Direito Processual Civil",
    "Direito Penal", "Direito Processual Penal", "Direito Administrativo",
    "Direito Tributário", "Direito Empresarial", "Direito do Consumidor",
    "Direito Ambiental", "Direito da Infância e Juventude", "Direitos Humanos",
    "Direito Eleitoral", "Direito Previdenciário", "Outro",
]
TAXONOMIAS["defensoria"] = [
    "Direito Constitucional", "Direitos Humanos", "Direito Penal",
    "Direito Processual Penal", "Execução Penal", "Direito Civil",
    "Direito Processual Civil", "Direito da Infância e Juventude",
    "Direito do Consumidor", "Direito Administrativo",
    "Direito Institucional da Defensoria Pública", "Direito Ambiental",
    "Direitos Difusos e Coletivos", "Direito Previdenciário", "Outro",
]
TAXONOMIAS["trabalho"] = [
    "Direito do Trabalho", "Direito Processual do Trabalho",
    "Direito Constitucional", "Direito Civil", "Direito Processual Civil",
    "Direito Administrativo", "Direito Previdenciário", "Direitos Humanos",
    "Direito Empresarial", "Direito Internacional do Trabalho", "Outro",
]


def taxonomia_do_cargo(cargo: str) -> list[str]:
    """A taxonomia sai do CARGO, nao de uma variavel global.

    Com varios concursos no mesmo CSV, uma lista unica ofereceria "Direito
    Institucional do Ministerio Publico" a um candidato a Juiz do Trabalho e
    nenhuma materia trabalhista a ele; o modelo entao forca a classificacao para
    a opcao mais proxima, e a agregacao por materia deixa de significar algo.
    """
    c = (cargo or "").lower()
    if "delegado" in c:
        return TAXONOMIAS["delegado"]
    if "defensor" in c:
        return TAXONOMIAS["defensoria"]
    if "trabalho" in c:
        return TAXONOMIAS["trabalho"]
    if "juiz" in c or "magistrad" in c:
        return TAXONOMIAS["magistratura"]
    return TAXONOMIAS["mp"]      # promotor e afins


# união de todas: a validação aceita qualquer matéria; o filtro fino é feito no prompt
MATERIAS = sorted({m for lista in TAXONOMIAS.values() for m in lista})
print(f"{len(TAXONOMIAS)} taxonomias | {len(MATERIAS)} matérias no total")
for v in VIDEOS:
    print(f"  {v['video_id']:9} {v['cargo'][:30]:30} -> "
          f"{len(taxonomia_do_cargo(v['cargo']))} matérias")
DIFICULDADES = {"facil", "media", "dificil"}
REACOES = {"aprovou", "parcial", "corrigiu", "insistiu", "neutro", "sem_resposta"}

SYSTEM_PARES = """Você extrai pares pergunta-resposta de um bloco de arguição de prova oral para o cargo de {cargo} ({banca}). Um examinador conduz; o candidato responde.

TAREFA
Receba os turnos numerados do bloco e devolva os pares. Para cada par, informe QUAIS ÍNDICES formam a pergunta e quais formam a resposta. Nunca reescreva o texto dos turnos.

COMO AGRUPAR
- Uma pergunta pode ocupar vários turnos do examinador, separados por um reconhecimento curto do candidato ("Sim, excelência"). Junte os índices do examinador num mesmo par e NÃO trate o reconhecimento como resposta.
- Uma resposta pode ocupar vários turnos do candidato, separados por um "certo" do examinador. Junte os índices do candidato.
- Turno do examinador que NÃO pergunta nada (saudação, "passo a palavra", comentário sobre a resposta anterior) não abre par novo.
- Se o candidato nunca respondeu, devolva "resposta": [] e reacao_examinador: "sem_resposta".
- Blocos que são só saudação ou passagem de palavra: devolva {{"pares": []}}.

CLASSIFICAÇÃO
- materia: EXATAMENTE um valor da lista em "materias_permitidas".
- tema: área dentro da matéria. Poucas palavras.
- subtema: o ponto específico cobrado.
- dificuldade:
  * "facil"   = conceito ou definição direta, resposta esperada curta.
  * "media"   = exige fundamentação legal, citação de artigo ou aplicação a caso simples.
  * "dificil" = exige divergência doutrinária, posição de tribunal, tese minoritária ou caso concreto complexo.

REAÇÃO DO EXAMINADOR (o que ele diz DEPOIS da resposta, dentro do bloco)
- "aprovou"      = confirma sem ressalva.
- "parcial"      = aceita mas complementa ou pede mais.
- "corrigiu"     = aponta erro ou dá a resposta certa.
- "insistiu"     = repete ou reformula a mesma pergunta.
- "neutro"       = passa adiante sem sinalizar.
- "sem_resposta" = o candidato não respondeu.

RESUMO
resumo_resposta: 1 a 2 frases sobre o que o candidato EFETIVAMENTE respondeu. Não corrija, não complemente. Se ele errou, o resumo registra o que ele disse.

SAÍDA
JSON puro, sem markdown:
{{"pares": [{{"pergunta": [0, 2], "resposta": [3], "materia": "Direito Penal", "tema": "...", "subtema": "...", "dificuldade": "media", "reacao_examinador": "aprovou", "resumo_resposta": "..."}}]}}
Índices de "pergunta" devem ser turnos do examinador; os de "resposta", do candidato."""


def montar_blocos(turnos):
    blocos, atual = [], None
    for i, t in enumerate(turnos):
        if t["papel"] == "examinador" and (atual is None or t["falante"] != atual["falante"]):
            atual = {"falante": t["falante"], "turnos": []}
            blocos.append(atual)
        if atual is not None:
            atual["turnos"].append({**t, "idx_video": i})
    return [b for b in blocos if len(b["turnos"]) > 1]   # 1 turno = passagem de palavra


blocos = []
for v in VIDEOS:
    vid = v["video_id"]
    for j, b in enumerate(montar_blocos(corpus[vid]["turnos"])):
        blocos.append({"bloco_id": f"{vid}_b{j:02d}", "video_id": vid,
                       "falante_examinador": b["falante"], "turnos": b["turnos"]})
print(f"{len(blocos)} blocos de arguição")

arq_brutos = DIR_BANCO / "pares_brutos.json"
pares_brutos = ({} if FORCAR or not arq_brutos.exists()
                else json.loads(arq_brutos.read_text(encoding="utf-8")))
por_video = {v["video_id"]: v for v in VIDEOS}

reqs_pares = []
for b in blocos:
    if b["bloco_id"] in pares_brutos and "erro" not in pares_brutos[b["bloco_id"]]:
        continue
    v = por_video[b["video_id"]]
    materias = taxonomia_do_cargo(v["cargo"])       # taxonomia por concurso
    user = json.dumps({"materias_permitidas": materias,
                       "turnos": [{"i": i, "papel": t["papel"], "texto": t["texto"]}
                                  for i, t in enumerate(b["turnos"])]}, ensure_ascii=False)
    reqs_pares.append({
        "custom_id": "pares_" + hash_curto([VERSAO_PROMPT, MODELO_LLM, b["bloco_id"],
                                            [t["texto"] for t in b["turnos"]]]),
        "system": SYSTEM_PARES.format(cargo=v["cargo"], banca=v["banca"]),
        "user": user, "ids": None, "_bloco": b["bloco_id"]})

print(f"etapa 6: {len(reqs_pares)} blocos a processar", flush=True)
res_pares, falhas_lote = executar_lote(reqs_pares, "pares", max_tokens=16000)
falhas_pares = [f["custom_id"] for f in falhas_lote]
for r in reqs_pares:
    if r["custom_id"] in res_pares:
        pares_brutos[r["_bloco"]] = res_pares[r["custom_id"]]

arq_brutos.write_text(json.dumps(pares_brutos, ensure_ascii=False, indent=1), encoding="utf-8")
n_pares = sum(len(x.get("pares", [])) for x in pares_brutos.values())
print(f"{len(pares_brutos)} blocos processados, {n_pares} pares brutos, "
      f"{len(falhas_pares)} falha(s)")
# Falha silenciosa aqui já deixou 2% do corpus de fora sem ninguém notar.
assert not falhas_pares, (f"{len(falhas_pares)} bloco(s) falharam: {falhas_pares[:5]}. "
                          "Corrija a causa e rode de novo: o cache guarda o que deu certo.")

### 7.1 Montar o banco, validando índice a índice

O texto é reconstruído **a partir do corpus**, pelos índices que o modelo devolveu.
Um par só entra se todo índice existir no bloco, os de pergunta forem de turnos do
examinador, os de resposta forem do candidato e a matéria estiver na taxonomia.

A célula recusa qualquer par cujo texto não venha literalmente dos turnos. Descarte
alto significa prompt ruim, não corpus ruim.

In [ ]:
def juntar(turnos, idxs):
    sel = [turnos[i] for i in idxs]
    return {"texto": " ".join(t["texto"] for t in sel),
            "inicio": min(t["inicio"] for t in sel), "fim": max(t["fim"] for t in sel),
            "n_palavras": sum(t["n_palavras"] for t in sel),
            "prob_media": round(float(np.mean([t["prob_media"] for t in sel])), 3),
            "idx": [t["idx_video"] for t in sel]}


banco, descartados = [], []
for b in blocos:
    turnos, n = b["turnos"], len(b["turnos"])
    v = por_video[b["video_id"]]
    for k, p in enumerate(pares_brutos.get(b["bloco_id"], {}).get("pares", [])):
        qi = [int(i) for i in p.get("pergunta", [])]
        ri = [int(i) for i in p.get("resposta", [])]
        motivo = None
        if not qi:
            motivo = "pergunta vazia"
        elif any(not 0 <= i < n for i in qi + ri):
            motivo = "indice fora do bloco"
        elif any(turnos[i]["papel"] != "examinador" for i in qi):
            motivo = "pergunta com turno de candidato"
        elif any(turnos[i]["papel"] != "candidato" for i in ri):
            motivo = "resposta com turno de examinador"
        elif p.get("materia") not in MATERIAS:
            motivo = f"materia invalida: {p.get('materia')}"
        if motivo:
            descartados.append({"bloco_id": b["bloco_id"], "motivo": motivo}); continue

        perg = juntar(turnos, sorted(qi))
        resp = juntar(turnos, sorted(ri)) if ri else None
        banco.append({
            "pergunta_id": f"{b['bloco_id']}_p{k:02d}", "video_id": b["video_id"],
            "banca": v["banca"], "cargo": v["cargo"], "edicao": v["edicao"],
            "uf": v["uf"],
            "ano": v["ano"], "grupo_tematico": v["grupo_tematico"],
            "colocacao": v["colocacao"],
            "materia": p["materia"], "tema": str(p.get("tema", ""))[:120],
            "subtema": str(p.get("subtema", ""))[:120],
            "dificuldade": p.get("dificuldade") if p.get("dificuldade") in DIFICULDADES else "media",
            "reacao_examinador": p.get("reacao_examinador") if p.get("reacao_examinador") in REACOES else "neutro",
            "inicio": perg["inicio"], "fim": (resp or perg)["fim"],
            "pergunta": perg["texto"], "resposta": resp["texto"] if resp else "",
            "resumo_resposta": str(p.get("resumo_resposta", ""))[:600],
            "prob_media_resposta": resp["prob_media"] if resp else None,
            "idx_pergunta": perg["idx"], "idx_resposta": resp["idx"] if resp else [],
        })

banco.sort(key=lambda x: (x["video_id"], x["inicio"]))

# o texto tem de ser IDENTICO a juncao dos turnos de origem
falhou = []
for r in banco:
    turnos = corpus[r["video_id"]]["turnos"]
    for campo, idxs in (("pergunta", r["idx_pergunta"]), ("resposta", r["idx_resposta"])):
        if r[campo] != " ".join(turnos[i]["texto"] for i in idxs):
            falhou.append((r["pergunta_id"], campo))
assert not falhou, f"{len(falhou)} trecho(s) nao batem com os turnos: {falhou[:5]}"

(DIR_BANCO / "banco_perguntas.json").write_text(
    json.dumps(banco, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"{len(banco)} pares válidos | {len(descartados)} descartados")
print(f"{2*len(banco)} trechos conferidos: todos vieram dos turnos originais")
if descartados:
    print(dict(Counter(d["motivo"] for d in descartados)))

## 8. Extrator determinístico de citações

Esta é a camada anti-alucinação que não depende de LLM nenhum. Cada citação vira uma
**tupla canônica**, para que a mesma norma dita de formas diferentes colida e normas
diferentes **não** colidam:

```
("norma", "l7347")          a norma foi mencionada
("art",   "l7347", "5")     artigo 5 daquela norma
("sumula","stj",  "231")    súmula, com o tribunal quando explícito
```

Assim "art. 5º da Lei 7.347/1985" e "artigo 5º da Lei da Ação Civil Pública" geram a
**mesma** tupla, e "art. 25 do Código Penal" e "art. 25 do Código Penal **português**"
geram tuplas **diferentes**: era esse o falso negativo da versão que comparava só o
número do artigo.

É com isso que a etapa 10 confere, contra a transcrição, todo dispositivo que o modelo
numerou. A origem que o modelo declara não é aceita: é verificada.

In [ ]:
# Camada 3: extrator determinístico de citações. Cada citação vira uma tupla canônica
# (norma, artigo, súmula, tema, enunciado), para a mesma norma dita de formas diferentes colidir
# e normas diferentes não: "art. 5º da Lei 7.347" = "art. 5º da Lei da ACP", mas "Código Penal
# português" não é o Código Penal. Leis e decretos-leis dividem o prefixo "l".

# Nome extenso -> slug canônico. A ordem importa: o mais específico primeiro.
NORMAS_NOMEADAS = [
    (r"constitui[çc][ãa]o\s+(?:estadual|do\s+estado)", "ce"),
    (r"constitui[çc][ãa]o(?:\s+(?:federal|da\s+rep[úu]blica)"
     r"(?:\s+federativa\s+do\s+brasil)?)?", "cf"),
    (r"c[óo]digo\s+de\s+processo\s+civil", "l13105"),
    (r"c[óo]digo\s+de\s+processo\s+penal", "l3689"),
    (r"c[óo]digo\s+de\s+defesa\s+do\s+consumidor", "l8078"),
    (r"c[óo]digo\s+tribut[áa]rio\s+nacional", "l5172"),
    (r"c[óo]digo\s+de\s+tr[âa]nsito\s+brasileiro", "l9503"),
    (r"c[óo]digo\s+penal\s+militar", "l1001"),
    (r"c[óo]digo\s+penal", "l2848"),
    (r"c[óo]digo\s+civil", "l10406"),
    (r"c[óo]digo\s+comercial", "l556"),
    (r"consolida[çc][ãa]o\s+das\s+leis\s+do\s+trabalho", "l5452"),
    (r"estatuto\s+da\s+crian[çc]a\s+e\s+do\s+adolescente", "l8069"),
    (r"estatuto\s+do\s+idoso|estatuto\s+da\s+pessoa\s+idosa", "l10741"),
    (r"estatuto\s+da\s+pessoa\s+com\s+defici[êe]ncia"
     r"|lei\s+brasileira\s+de\s+inclus[ãa]o", "l13146"),
    (r"estatuto\s+do\s+desarmamento", "l10826"),
    (r"estatuto\s+da\s+cidade", "l10257"),
    (r"lei\s+de\s+execu[çc][ãa]o\s+penal", "l7210"),
    (r"lei\s+de\s+introdu[çc][ãa]o[^.;]{0,45}?direito\s+brasileiro", "l4657"),
    (r"lei\s+d[ae]\s+a[çc][ãa]o\s+civil\s+p[úu]blica", "l7347"),
    (r"lei\s+d[ae]\s+a[çc][ãa]o\s+popular", "l4717"),
    (r"lei\s+de\s+improbidade(?:\s+administrativa)?", "l8429"),
    (r"lei\s+(?:de\s+)?(?:anti)?drogas|lei\s+de\s+t[óo]xicos", "l11343"),
    (r"lei\s+maria\s+da\s+penha", "l11340"),
    (r"lei\s+henry\s+borel", "l14344"),
    (r"lei\s+do\s+mandado\s+de\s+seguran[çc]a", "l12016"),
    (r"lei\s+dos\s+juizados\s+especiais", "l9099"),
    (r"lei\s+de\s+abuso\s+de\s+autoridade", "l13869"),
    (r"lei\s+anticorrup[çc][ãa]o|lei\s+da\s+empresa\s+limpa", "l12846"),
    (r"lei\s+de\s+mobilidade\s+urbana"
     r"|pol[íi]tica\s+nacional\s+de\s+mobilidade\s+urbana", "l12587"),
    (r"lei\s+de\s+licita[çc][õo]es\s+e\s+contratos\s+administrativos", "l14133"),
    (r"lei\s+da\s+ficha\s+limpa", "lc135"),
    (r"lei\s+de\s+responsabilidade\s+fiscal", "lc101"),
    (r"lei\s+das\s+elei[çc][õo]es", "l9504"),
    (r"lei\s+org[âa]nica\s+nacional\s+do\s+minist[ée]rio\s+p[úu]blico", "l8625"),
    (r"lei\s+de\s+crimes\s+ambientais", "l9605"),
    (r"lei\s+de\s+organiza[çc][ãa]o\s+criminosa", "l12850"),
]
RX_NOMEADAS = [(re.compile(p, re.I), slug) for p, slug in NORMAS_NOMEADAS]

# Siglas: casadas com sensibilidade a maiúsculas, senão "cf." (confer) e o "cc"
# de "cc." viram norma.
SIGLAS = {
    "CF": "cf", "CRFB": "cf", "CE": "ce", "CPC": "l13105", "CPP": "l3689",
    "CP": "l2848", "CPM": "l1001", "CC": "l10406", "CDC": "l8078", "CTN": "l5172",
    "CTB": "l9503", "CLT": "l5452", "ECA": "l8069", "LEP": "l7210",
    "LINDB": "l4657", "LACP": "l7347", "LIA": "l8429", "LONMP": "l8625",
    "LRF": "lc101", "LOA": "loa",
}
RX_SIGLAS = re.compile(r"\b(" + "|".join(sorted(SIGLAS, key=len, reverse=True)) + r")\b")

# Norma identificada por número. O prefixo separa espaços de numeração distintos.
PREFIXO_TIPO = {
    "lei complementar": "lc", "emenda constitucional": "ec",
    "medida provisoria": "mp", "decreto-lei": "l", "decreto lei": "l",
    "decreto legislativo": "dleg", "decreto": "d", "resolucao": "res",
    "instrucao normativa": "in", "portaria": "port", "provimento": "prov",
    "lei": "l",
}
RX_NORMA_NUM = re.compile(
    r"\b(?P<tipo>lei\s+complementar|emenda\s+constitucional|medida\s+provis[óo]ria"
    r"|decreto[\s-]+lei|decreto\s+legislativo|decreto|resolu[çc][ãa]o"
    r"|instru[çc][ãa]o\s+normativa|portaria|provimento|lei)"
    # o órgão é opcional ("Resolução CNMP 243"); o lookahead impede que a
    # preposição de "lei complementar DE 2001" seja lida como número da norma
    r"(?P<orgao>(?:\s+(?:d[aeo]s?\s+)?(?!d[eo]\b)[^\W\d_]{2,10}\b)?)"
    r"[\s,]*(?:n[º°ºo]?\.?\s*)?"
    r"(?P<num>\d{1,3}(?:\.\d{3})+|\d{1,6})(?:\s*/\s*\d{2,4})?", re.I)

# Adjetivo de nacionalidade logo após a norma: "Código Penal português" não é o
# nosso Código Penal e não pode casar com ele.
RX_ESTRANGEIRA = re.compile(
    r"^\W{0,3}(portugu[êe]s|espanhol|alem[ãa]o|italiano|franc[êe]s|argentino|chileno"
    r"|de\s+portugal|da\s+espanha|da\s+alemanha|da\s+it[áa]lia|da\s+fran[çc]a)", re.I)

NUM_ART = r"\d[\d.]*\d|\d"
RX_ARTIGOS = re.compile(
    rf"\b(?:artigos?|arts?)\.?\s*"
    rf"(?P<lista>(?:{NUM_ART})[º°ªo]?(?:\s*-\s*[A-Za-z]\b)?"
    rf"(?:\s*(?:,|;|\be\b|\bou\b|\ba\b)\s*(?:{NUM_ART})[º°ªo]?(?:\s*-\s*[A-Za-z]\b)?)*)",
    re.I)
RX_UM_ART = re.compile(rf"(?P<n>{NUM_ART})[º°ªo]?(?:\s*-\s*(?P<suf>[A-Za-z])\b)?")

RX_SUMULA = re.compile(
    r"\bs[úu]mula\s+(?P<vinc>vinculante\s+)?(?:n[º°ºo]?\.?\s*)?(?P<n>\d+)"
    r"(?:\s*(?:d[oa]\s+)?(?P<trib>STF|STJ|TST|TSE|STM|TNU))?", re.I)
RX_TEMA = re.compile(
    r"\btema\s+(?:de\s+)?(?:repercuss[ãa]o\s+geral\s+|repetitiv[oa]\s+)?"
    r"(?:n[º°ºo]?\.?\s*)?(?P<n>\d+)(?:\s*(?:d[oa]\s+)?(?P<trib>STF|STJ))?", re.I)
RX_ENUNCIADO = re.compile(
    r"\benunciado\s+(?:n[º°ºo]?\.?\s*)?(?P<n>\d+)"
    r"(?:\s*(?:d[aoe]s?\s+)?(?P<org>[A-ZÀ-Ý]{2,10}))?")

# Distância máxima, em caracteres, entre um artigo e a norma a que ele se liga.
JANELA_NORMA = 130


def _num(s):
    return re.sub(r"[.\s]", "", s)


def _localizar_normas(t):
    """[(inicio, fim, slug)] de toda norma mencionada, ordenado por posição."""
    achados = []
    for rx, slug in RX_NOMEADAS:
        for m in rx.finditer(t):
            achados.append((m.start(), m.end(), slug))
    for m in RX_SIGLAS.finditer(t):
        achados.append((m.start(), m.end(), SIGLAS[m.group(1)]))
    for m in RX_NORMA_NUM.finditer(t):
        tipo = re.sub(r"[\s-]+", " ", m.group("tipo").lower())
        tipo = tipo.replace("ó", "o").replace("ç", "c").replace("ã", "a")
        orgao = (m.group("orgao") or "").strip()
        orgao = orgao if orgao.isupper() and len(orgao) >= 2 else ""
        pref = PREFIXO_TIPO.get(tipo, "l")
        slug = f"{pref}{orgao.lower()}_{_num(m.group('num'))}" if orgao \
            else f"{pref}{_num(m.group('num'))}"
        achados.append((m.start(), m.end(), slug))

    # sufixo de nacionalidade -> norma estrangeira, que nunca casa com a nacional
    marcados = []
    for ini, fim, slug in achados:
        if RX_ESTRANGEIRA.match(t[fim:fim + 30]):
            slug += "_estrangeiro"
        marcados.append((ini, fim, slug))

    # descarta menções contidas noutra maior ("Constituição" dentro de "Constituição Estadual")
    marcados.sort(key=lambda x: (x[0], -(x[1] - x[0])))
    saida = []
    for ini, fim, slug in marcados:
        if any(a <= ini and fim <= b for a, b, _ in saida):
            continue
        saida.append((ini, fim, slug))
    return sorted(saida)


def _norma_do_artigo(pos_ini, pos_fim, normas):
    """Norma mais próxima à frente do artigo; se não houver, a mais próxima atrás."""
    frente = [(a - pos_fim, s) for a, _, s in normas if a >= pos_fim]
    if frente:
        d, slug = min(frente)
        if d <= JANELA_NORMA:
            return slug
    atras = [(pos_ini - b, s) for _, b, s in normas if b <= pos_ini]
    if atras:
        d, slug = min(atras)
        if d <= JANELA_NORMA:
            return slug
    return None


def extrair_citacoes(texto):
    """Conjunto de tuplas canônicas de citação encontradas no texto."""
    t = texto or ""
    normas = _localizar_normas(t)
    out = {("norma", slug) for _, _, slug in normas}

    for m in RX_ARTIGOS.finditer(t):
        slug = _norma_do_artigo(m.start(), m.end(), normas)
        for a in RX_UM_ART.finditer(m.group("lista")):
            n = _num(a.group("n")) + (a.group("suf") or "").lower()
            out.add(("art", slug, n))

    for m in RX_SUMULA.finditer(t):
        orig = "vinculante" if m.group("vinc") else (
            m.group("trib").lower() if m.group("trib") else None)
        out.add(("sumula", orig, m.group("n")))
    for m in RX_TEMA.finditer(t):
        out.add(("tema", m.group("trib").lower() if m.group("trib") else None, m.group("n")))
    for m in RX_ENUNCIADO.finditer(t):
        out.add(("enunciado", m.group("org").lower() if m.group("org") else None, m.group("n")))
    return out


def ancorada(cit, ancora):
    """A citação encontra respaldo na transcrição da prova?"""
    if cit in ancora:
        return True
    if cit[0] != "art":
        return False
    _, norma, num = cit
    if norma is None:
        return ("art", None, num) in ancora
    # a fala raramente junta artigo e diploma na mesma frase: aceita o artigo
    # solto da transcrição desde que aquela norma tenha sido citada em algum ponto
    return ("art", None, num) in ancora and ("norma", norma) in ancora


ROTULOS = {
    "cf": "CF", "ce": "CE", "l13105": "CPC", "l3689": "CPP", "l2848": "CP",
    "l1001": "CPM", "l10406": "CC", "l8078": "CDC", "l5172": "CTN", "l8069": "ECA",
    "l7210": "LEP", "l9503": "CTB", "l5452": "CLT", "l4657": "LINDB",
}
NOME_TIPO = {"l": "Lei", "lc": "LC", "ec": "EC", "mp": "MP", "d": "Decreto",
             "dleg": "Dec. Legislativo", "res": "Resolução", "in": "IN",
             "port": "Portaria", "prov": "Provimento"}


def rotulo_norma(slug):
    if slug.endswith("_estrangeiro"):
        return rotulo_norma(slug[:-len("_estrangeiro")]) + " (estrangeiro)"
    if slug in ROTULOS:
        return ROTULOS[slug]
    m = re.match(r"(dleg|prov|port|res|in|lc|ec|mp|d|l)([a-z]*)_?(\d+)$", slug)
    if not m:
        return slug
    tipo, orgao, num = m.groups()
    num = f"{int(num):,}".replace(",", ".") if len(num) > 3 else num
    return " ".join(x for x in (NOME_TIPO.get(tipo, tipo), orgao.upper(), num) if x)


def rotulo(cit):
    if cit[0] == "norma":
        return rotulo_norma(cit[1])
    if cit[0] == "art":
        alvo = rotulo_norma(cit[1]) if cit[1] else "norma não identificada"
        return f"art. {cit[2]}, {alvo}"
    nome = {"sumula": "Súmula", "tema": "Tema", "enunciado": "Enunciado"}[cit[0]]
    if cit[0] == "sumula" and cit[1] == "vinculante":
        return f"Súmula Vinculante {cit[2]}"
    return f"{nome} {cit[2]}" + (f" do {cit[1].upper()}" if cit[1] else "")

## 9. Estruturar, revisar e corrigir

Três passadas, e é aqui que a resposta errada do candidato vira gabarito correto.

| passada | o que faz |
|---|---|
| **p1 estruturação** | janela deslizante de 6 itens + 3 de contexto → pergunta autocontida + gabarito |
| **p2 revisor cego** | recebe só pergunta+resposta, **sem** a transcrição, e procura erro jurídico |
| **p3 regeneração** | reescreve apenas o que o revisor marcou como grave ou suspeito |

As cinco camadas anti-alucinação:

1. **abstenção explícita permitida** → `gabarito_status` pode ser "abstido"
2. **proveniência por dispositivo** → `origem_verificada`, conferida contra a transcrição
3. **extrator determinístico** (etapa 8) → cobre o gabarito **e** os fundamentos
4. **revisor cego** → não vê a fonte, avalia só o conteúdo
5. **regeneração dirigida** → com os apontamentos do revisor tratados como hipótese

> **O gabarito é a resposta que um candidato nota máxima daria, não a fala limpa do
> candidato.** A transcrição é âncora do recorte: indica qual doutrina e quais
> dispositivos o examinador queria ouvir. Onde a fala estiver errada ou incompleta, o
> gabarito traz o correto, sem comentar o erro e sem mencionar o candidato.

In [ ]:
SYSTEM_ESTRUTURA = """Você é um examinador experiente de provas orais para o cargo de {cargo} ({banca}) e revisor de material didático para concursos.

Você recebe trechos de transcrições reais de provas orais (Whisper + diarização). Converta cada par pergunta/resposta em um item de banco de questões utilizável por um simulador.

PERGUNTA
1. Reescreva de modo AUTOCONTIDO: precisa fazer sentido isolada. Use o contexto para resolver referências ("E para o senhor?", "Integralmente?").
2. Preserve a intenção, o recorte jurídico e o nível de exigência. NÃO facilite, NÃO amplie, NÃO transforme pergunta específica em genérica.
3. Remova ruído oral, saudações e feedback ao turno anterior. Corrija erros óbvios de transcrição em nomes de leis, números de artigos e termos técnicos.
4. Mantenha o tratamento formal de prova oral ("o senhor" / "a senhora").

RESPOSTA (é gabarito, não transcrição)
5. A resposta que você escreve é o GABARITO: a resposta que um candidato nota máxima daria. NÃO é a fala do candidato limpa.
6. Use a transcrição como ÂNCORA DO RECORTE: ela indica qual doutrina, qual corrente e quais dispositivos o examinador queria ouvir; e o seu conhecimento jurídico para garantir a correção. Onde a fala estiver errada ou incompleta, escreva o correto sem comentar o erro e sem mencionar o candidato.
7. O campo reacao_examinador é sinal forte: "corrigiu" e "insistiu" indicam que a resposta esperada não é a que o candidato deu.
8. Prosa contínua, 3 a 8 frases, sem tópicos, sem "Excelência", sem referência à banca ou ao candidato. Responda a pergunta por inteiro. Se houver divergência doutrinária relevante, explicite as correntes.

PRECISÃO: leia com atenção
9. Você PODE se abster. Use gabarito_status:
   - "completo": segurança jurídica em todo o conteúdo;
   - "parcial": responde o núcleo mas omite parte por insegurança;
   - "abstido": transcrição corrompida ou sem segurança; devolva resposta vazia.
   Um gabarito incompleto é aceitável. Um gabarito confiante e errado é o pior resultado possível.
10. NUNCA cite dispositivo, súmula ou precedente do qual não tenha certeza do número. Se lembra do conteúdo mas não do número, descreva a regra sem numerá-la.
11. Em fundamentos_legais, cada item tem DOIS campos distintos:
    - "ref": o texto do dispositivo em si ("art. 5º, LX, da Constituição Federal",
      "Súmula Vinculante 11", "ADPF 130"). NUNCA escreva "transcricao" nem "modelo"
      neste campo: são valores do campo "origem", e confundi-los invalida o item.
    - "origem": de onde ele veio, e só aceita dois valores:
      "transcricao" = o dispositivo aparece na pergunta ou na resposta transcrita;
      "modelo"      = você o acrescentou por conhecimento próprio.
    Nao copie o exemplo do formato como se fosse conteudo: ele e so ilustracao.
12. confianca de 0.0 a 1.0. Use < 0.6 quando a pergunta seguir ambígua, a transcrição estiver corrompida, ou não tiver segurança jurídica.
13. A colocação do candidato é CALIBRADOR, não garantia. Candidatos bem colocados também erram. Mantida a discordância, escreva o juridicamente correto.
14. descartar=true apenas quando o turno não é pergunta avaliável (encerramento, comentário, ruído).

Responda SOMENTE com JSON válido, sem markdown:
{{"itens": [{{"id": "...", "pergunta": "...", "resposta": "...", "pontos_chave": ["..."], "fundamentos_legais": [{{"ref": "<o dispositivo, p.ex. art. N da Lei X>", "origem": "transcricao"}}], "tipo": "principal|desdobramento", "gabarito_status": "completo|parcial|abstido", "confianca": 0.0, "descartar": false, "observacao": ""}}]}}

itens deve conter EXATAMENTE um objeto por item de PARA_REESCREVER, na mesma ordem e com o mesmo id. Nunca inclua itens de CONTEXTO na saída."""

SYSTEM_REVISOR = """Você é um revisor técnico de material para concursos jurídicos. Recebe itens de banco de questões já prontos e procura ERRO JURÍDICO.

Você NÃO tem acesso à fonte de onde o item foi extraído. Avalie apenas pelo próprio conteúdo.

Para cada item verifique:
1. A resposta responde à pergunta feita, por inteiro?
2. Há afirmação juridicamente incorreta, desatualizada ou revogada?
3. Os dispositivos citados existem e correspondem ao conteúdo alegado? Número de artigo, de lei e de súmula trocados são o erro mais comum: liste em dispositivos_suspeitos todo dispositivo cujo número você não confirme com segurança.
4. Há afirmação apresentada como pacífica que na verdade é controvertida?

veredito:
  "ok": sem problema relevante;
  "ajustes": impreciso, incompleto ou com dispositivo duvidoso;
  "grave": contém erro jurídico que induziria o candidato a erro.

Não reescreva o item, apenas aponte. Na dúvida sobre um número, marque como suspeito: falso positivo custa uma revisão humana, falso negativo entra no gabarito.

Responda SOMENTE com JSON válido, sem markdown:
{"itens": [{"id": "...", "veredito": "ok|ajustes|grave", "problemas": ["..."], "dispositivos_suspeitos": ["..."], "confianca": 0.0}]}

Um objeto por item recebido, mesma ordem, mesmo id."""

SYSTEM_CORRECAO = """Você é um examinador experiente de provas orais para o cargo de {cargo} ({banca}). Um item de banco de questões foi reprovado por um revisor técnico. Reescreva-o.

1. Trate os apontamentos do revisor como hipóteses qualificadas, não como verdade. Se o revisor estiver certo, corrija. Se estiver errado, mantenha e explique em observacao.
2. A transcrição original é a âncora do RECORTE, nunca a fonte da correção jurídica: a fala do candidato pode ser justamente o que estava errado.
3. Remova todo dispositivo cujo número você não confirme. Descrever a regra sem numerá-la é melhor que numerá-la errado.
4. Se depois de reavaliar não tiver segurança, use gabarito_status "parcial" ou "abstido".
5. Mesmas regras de forma: prosa contínua, 3 a 8 frases, sem tópicos, sem menção ao candidato ou à banca.

Responda SOMENTE com JSON válido, no mesmo formato de sempre:
{{"itens": [{{"id": "...", "pergunta": "...", "resposta": "...", "pontos_chave": ["..."], "fundamentos_legais": [{{"ref": "...", "origem": "transcricao|modelo"}}], "tipo": "principal|desdobramento", "gabarito_status": "completo|parcial|abstido", "confianca": 0.0, "descartar": false, "observacao": ""}}]}}"""

In [ ]:
# Siglas vêm de nomes.py: tipo de órgão + UF (TJMT, MPBA, PCSP).
from nomes import sigla_banca, sigla_cargo, slug


def montar_provas(dados):
    """1 video = 1 prova.

    O prova_id carrega UF, banca e cargo, nao so o ano. Com 28 videos de 10 bancas
    diferentes, `prova_2024_01` nao diz de qual concurso e, e dois estados com prova
    no mesmo ano colidiriam no mesmo id.
    """
    por_video = defaultdict(list)
    for x in dados:
        por_video[x["video_id"]].append(x)
    contador, provas = defaultdict(int), []
    for vid in sorted(por_video, key=lambda s: int(s.split("_")[1])):
        itens = sorted(por_video[vid], key=lambda x: x["inicio"])
        p0 = itens[0]
        ano = p0["ano"] or None          # ano desconhecido nao vira "0" no id
        concurso_id = (f"{p0['uf']}_{sigla_banca(p0['banca'], p0['uf'])}_"
                       f"{sigla_cargo(p0['cargo'])}")
        chave = (concurso_id, ano)
        contador[chave] += 1
        citacoes = set()
        for x in itens:
            citacoes |= extrair_citacoes(x["pergunta"] + " " + x["resposta"])
        provas.append({
            "prova_id": f"{concurso_id}_{ano or 'sd'}_{contador[chave]:02d}",
            "concurso_id": concurso_id,
            "video_id": vid, "uf": p0["uf"], "ano": ano, "edicao": p0["edicao"],
            "banca": p0["banca"], "cargo": p0["cargo"],
            "grupo_tematico": p0["grupo_tematico"], "colocacao": p0["colocacao"],
            "itens": itens, "citacoes_prova": citacoes})
    return provas


def janelas(itens):
    return [(itens[max(0, i - CONTEXTO):i], itens[i:i + JANELA])
            for i in range(0, len(itens), JANELA)]


def montar_user_estrutura(ctx, alvos, prova):
    p = [f"PROVA: {prova['edicao'] or '-'} ({prova['ano'] or '-'}): "
         f"{prova['banca']}, {prova['cargo']}"]
    if prova.get("grupo_tematico"):
        p.append(f"GRUPO TEMÁTICO: {prova['grupo_tematico']}")
    c = prova.get("colocacao")
    p.append(f"COLOCAÇÃO DO CANDIDATO: {c}º lugar (calibrador, não garantia)" if c
             else "COLOCAÇÃO DO CANDIDATO: não informada")
    p.append("")
    if ctx:
        p.append("CONTEXTO (turnos anteriores, apenas para resolver referências; "
                 "NÃO devem aparecer na saída):")
        for it in ctx:
            p.append(f"- [{it['materia']} / {it['tema']}] Examinador: {it['pergunta']}")
            p.append(f"  Candidato (resumo): {it['resumo_resposta']}")
    else:
        p.append("CONTEXTO: (início da prova)")
    p.append("")
    p.append("PARA_REESCREVER:")
    for it in alvos:
        p.append(json.dumps({"id": it["pergunta_id"], "materia": it["materia"],
                             "tema": it["tema"], "subtema": it["subtema"],
                             "dificuldade": it["dificuldade"],
                             "reacao_examinador": it["reacao_examinador"],
                             "pergunta_transcrita": it["pergunta"],
                             "resposta_transcrita": it["resposta"],
                             "resumo_resposta": it["resumo_resposta"]},
                            ensure_ascii=False))
    p.append("")
    p.append(f"Retorne o JSON com {len(alvos)} itens em itens.")
    return "\n".join(p)


provas = montar_provas(banco)
print(f"{len(provas)} prova(s):")
for p in provas:
    print(f"  {p['prova_id']:16} {p['video_id']:9} {len(p['itens']):3} pares")

# p1: estruturação
reqs_p1, mapa_p1 = [], []
for prova in provas:
    for ctx, alvos in janelas(prova["itens"]):
        cid = "p1_" + hash_curto([VERSAO_PROMPT, MODELO_LLM, prova["prova_id"],
                                  [c["pergunta_id"] for c in ctx],
                                  [[a["pergunta_id"], a["pergunta"], a["resposta"]]
                                   for a in alvos]])
        reqs_p1.append({"custom_id": cid,
                        "system": SYSTEM_ESTRUTURA.format(cargo=prova["cargo"],
                                                          banca=prova["banca"]),
                        "user": montar_user_estrutura(ctx, alvos, prova),
                        "ids": [a["pergunta_id"] for a in alvos]})
        mapa_p1.append(cid)

print(f"\np1 estruturação: {len(reqs_p1)} janelas")
res_p1, falhas_p1 = executar_lote(reqs_p1, "p1")
assert not falhas_p1, f"p1 falhou: {falhas_p1[:3]}"

respostas = {}
for cid in mapa_p1:
    for it in res_p1.get(cid, {}).get("itens", []):
        respostas[it["id"]] = it
print(f"   {len(respostas)} itens estruturados")

### 9.1 Montagem do item e auditoria de citações

`auditar_citacoes` confere contra a transcrição **tudo** o que o modelo numerou:
tanto o texto do gabarito quanto os `fundamentos_legais`. A origem que o modelo
declarou por dispositivo deixa de ser aceita no escuro e passa a ser **verificada**.

A distinção que importa na fila de revisão: dispositivo numerado no texto do
gabarito sem respaldo na prova é risco real e vira motivo de revisão; fundamento
marcado "modelo" é, por definição, acréscimo do modelo: vira métrica, não fila.

In [ ]:
DIFICULDADE_EXT = {"facil": "fácil", "media": "média", "dificil": "difícil"}


def limpar(s):
    return re.sub(r"\s+", " ", (s or "")).strip()


def momento(ordem, total):
    r = ordem / max(total, 1)
    return "inicio" if r <= 0.25 else ("fim" if r > 0.75 else "meio")


def texto_embedding(it):
    return (f"Matéria: {it['materia']}\n"
            f"Tema: {it['tema']}\n"
            f"Dificuldade: {DIFICULDADE_EXT.get(it['dificuldade'], it['dificuldade'])}\n"
            f"Pergunta: {it['pergunta']}\n"
            f"Resposta: {it['resposta']}")


def auditar_citacoes(item, prova):
    anc = prova["citacoes_prova"]
    m = item["_meta"]

    def particionar(cits):
        soltos = {c for c in cits if c[0] == "art" and c[1] is None}
        return (sorted(rotulo(c) for c in cits - soltos if not ancorada(c, anc)),
                sorted(rotulo(c) for c in soltos if not ancorada(c, anc)))

    fora_resp, soltos = particionar(extrair_citacoes(item["resposta"]))
    fora_fund, divergentes, nao_verificaveis = set(), [], []
    n_conferidas = 0
    for f in m["fundamentos_legais"]:
        cits = extrair_citacoes(f["ref"])
        if not cits:
            f["origem_verificada"] = "nao_verificavel"
            nao_verificaveis.append(f["ref"]); continue
        n_conferidas += len(cits)
        fora = {c for c in cits if not ancorada(c, anc)}
        f["origem_verificada"] = "modelo" if fora else "transcricao"
        if f["origem"] == "transcricao" and f["origem_verificada"] == "modelo":
            divergentes.append(f["ref"])
        fora_fund |= fora
        soltos += [rotulo(c) for c in cits
                   if c[0] == "art" and c[1] is None and not ancorada(c, anc)]

    m["citacoes_nao_ancoradas"] = fora_resp
    m["auditoria_citacoes"] = {
        "n_citacoes_conferidas": n_conferidas + len(extrair_citacoes(item["resposta"])),
        "nao_ancoradas_resposta": fora_resp,
        "nao_ancoradas_fundamentos": sorted(rotulo(c) for c in fora_fund),
        "artigos_sem_norma": sorted(set(soltos)),
        "origem_divergente": divergentes,
        "refs_nao_verificaveis": nao_verificaveis,
    }
    return item


def aplicar_conteudo(item, llm_item, prova):
    # O modelo às vezes devolve {"ref": "transcricao"} (valor de `origem`): descartado, para não
    # virar citação vazia no gabarito.
    LIXO_REF = {"transcricao", "transcrição", "modelo", "ref", "n/a", "-"}
    fundamentos = []
    for f in (llm_item.get("fundamentos_legais") or []):
        ref = limpar(f.get("ref"))
        if not ref or ref.lower() in LIXO_REF or len(ref) < 4:
            continue
        fundamentos.append({"ref": ref, "origem": f.get("origem", "modelo"),
                            "origem_verificada": None})
    item["pergunta"] = limpar(llm_item.get("pergunta"))
    item["resposta"] = limpar(llm_item.get("resposta"))
    m = item["_meta"]
    m["pontos_chave"] = [limpar(p) for p in (llm_item.get("pontos_chave") or []) if limpar(p)]
    m["fundamentos_legais"] = fundamentos
    m["tipo"] = llm_item.get("tipo", m.get("tipo", "principal"))
    m["gabarito_status"] = llm_item.get("gabarito_status", "completo")
    m["confianca"] = float(llm_item.get("confianca") or 0.0)
    m["observacao"] = limpar(llm_item.get("observacao"))
    return auditar_citacoes(item, prova)


def montar_item(orig, llm_item, prova, ordem, total):
    item = {
        "uid": f"{prova['prova_id']}/q{ordem:04d}",
        "id": f"q{ordem:04d}", "prova_id": prova["prova_id"],
        "concurso_id": prova["concurso_id"], "uf": prova["uf"],
        "banca": prova["banca"], "cargo": prova["cargo"], "ordem": ordem,
        "materia": orig["materia"], "tema": orig["tema"],
        "dificuldade": orig["dificuldade"], "momento": momento(ordem, total),
        "pergunta": "", "resposta": "", "embedding_text": "",
        "_meta": {
            "subtema": orig["subtema"], "tipo": "principal", "pontos_chave": [],
            "fundamentos_legais": [], "gabarito_status": "completo", "confianca": 0.0,
            "citacoes_nao_ancoradas": [], "auditoria_citacoes": {},
            "colocacao_candidato": prova["colocacao"],
            "reacao_examinador": orig["reacao_examinador"], "observacao": "",
            "revisao_llm": None, "regenerado": False,
            "revisao_necessaria": True, "motivos_revisao": [],
            "origem": {
                "pergunta_id": orig["pergunta_id"], "video_id": orig["video_id"],
                "bloco_id": orig["pergunta_id"].rsplit("_p", 1)[0],
                "inicio": orig["inicio"], "fim": orig["fim"],
                "pergunta_transcrita": orig["pergunta"],
                "resposta_transcrita": orig["resposta"],
                "resumo_resposta": orig["resumo_resposta"],
                "prob_media_resposta": orig["prob_media_resposta"],
            },
        },
    }
    return aplicar_conteudo(item, llm_item, prova)


def finalizar_item(item, total):
    item["momento"] = momento(item["ordem"], total)
    item["embedding_text"] = texto_embedding(item)
    m = item["_meta"]
    rev = m.get("revisao_llm") or {}
    motivos = []
    if m["confianca"] < 0.6:
        motivos.append("confianca_baixa")
    if m["gabarito_status"] != "completo":
        motivos.append(f"gabarito_{m['gabarito_status']}")
    if len(item["resposta"].split()) < 15:
        motivos.append("resposta_curta")
    if not item["pergunta"]:
        motivos.append("pergunta_vazia")
    if m["citacoes_nao_ancoradas"]:
        motivos.append("citacao_nao_ancorada")
    if m["auditoria_citacoes"].get("artigos_sem_norma"):
        motivos.append("citacao_sem_norma")
    if rev.get("veredito") in ("ajustes", "grave") and not m["regenerado"]:
        motivos.append(f"revisor_{rev['veredito']}")
    if rev.get("dispositivos_suspeitos") and not m["regenerado"]:
        motivos.append("dispositivo_suspeito")
    if m["regenerado"]:
        motivos.append("regenerado_conferir")
    m["motivos_revisao"] = motivos
    m["revisao_necessaria"] = bool(motivos)
    return item


def precisa_regenerar(item):
    m = item["_meta"]
    rev = m.get("revisao_llm") or {}
    return bool(rev.get("veredito") == "grave" or rev.get("dispositivos_suspeitos")
                or (rev.get("veredito") == "ajustes" and m["confianca"] < 0.7))


# montagem inicial
por_prova, descartadas_prova = {}, {}
for prova in provas:
    finais, descartadas, ordem = [], [], 0
    for orig in prova["itens"]:
        llm_item = respostas.get(orig["pergunta_id"])
        if llm_item is None:
            continue
        if llm_item.get("descartar"):
            descartadas.append({"pergunta_id": orig["pergunta_id"],
                                "motivo": limpar(llm_item.get("observacao")) or "não avaliável",
                                "pergunta_transcrita": orig["pergunta"]})
            continue
        ordem += 1
        finais.append(montar_item(orig, llm_item, prova, ordem, len(prova["itens"])))
    por_prova[prova["prova_id"]] = finais
    descartadas_prova[prova["prova_id"]] = descartadas

todos = [i for p in provas for i in por_prova[p["prova_id"]]]
print(f"{len(todos)} questões montadas, "
      f"{sum(len(v) for v in descartadas_prova.values())} descartadas")

### 9.2 Revisor cego e regeneração

O revisor recebe **apenas** pergunta, resposta e fundamentos, nunca a transcrição.
Isso é deliberado: se ele visse a fonte, tenderia a validar o item contra ela em vez
de contra o direito, e o erro que veio da própria fala do candidato passaria batido.

Na regeneração, o veredito guardado descreve o texto **anterior**. Sem a marca
`obsoleta`, o revisor humano caçaria no gabarito novo um erro que já saiu.

In [ ]:
# p2: revisor cego
indice = {i["prova_id"] + "|" + i["id"]: i for i in todos}
reqs_p2 = []
for prova in provas:
    revisaveis = [i for i in por_prova[prova["prova_id"]] if i["resposta"]]
    for k in range(0, len(revisaveis), JANELA):
        lote = revisaveis[k:k + JANELA]
        payload = [{"id": x["id"], "materia": x["materia"], "tema": x["tema"],
                    "pergunta": x["pergunta"], "resposta": x["resposta"],
                    "fundamentos_legais": [f["ref"] for f in x["_meta"]["fundamentos_legais"]]}
                   for x in lote]
        cid = "p2_" + hash_curto([VERSAO_PROMPT, MODELO_LLM, prova["prova_id"], payload])
        reqs_p2.append({"custom_id": cid, "system": SYSTEM_REVISOR,
                        "user": "ITENS PARA REVISÃO:\n" +
                                "\n".join(json.dumps(x, ensure_ascii=False) for x in payload) +
                                f"\n\nRetorne o JSON com {len(payload)} itens em itens.",
                        "ids": [x["id"] for x in lote], "_prova": prova["prova_id"]})

print(f"p2 revisor cego: {len(reqs_p2)} lotes")
res_p2, falhas_p2 = executar_lote(reqs_p2, "p2")
assert not falhas_p2, f"p2 falhou: {falhas_p2[:3]}"

for r in reqs_p2:
    for rv in res_p2.get(r["custom_id"], {}).get("itens", []):
        alvo = indice.get(r["_prova"] + "|" + rv["id"])
        if alvo is None:
            continue
        alvo["_meta"]["revisao_llm"] = {
            "veredito": rv.get("veredito", "ok"),
            "problemas": [limpar(p) for p in (rv.get("problemas") or [])],
            "dispositivos_suspeitos": [limpar(p) for p in (rv.get("dispositivos_suspeitos") or [])],
            "confianca": float(rv.get("confianca") or 0.0)}
print("   " + str(dict(Counter((i["_meta"].get("revisao_llm") or {}).get("veredito")
                               for i in todos))))

# p3: regeneração
por_prova_obj = {p["prova_id"]: p for p in provas}
alvos_p3 = [(por_prova_obj[i["prova_id"]], i) for i in todos if precisa_regenerar(i)]
if alvos_p3:
    reqs_p3 = []
    for prova, item in alvos_p3:
        m = item["_meta"]; r = m.get("revisao_llm") or {}
        user = "\n".join([
            "ITEM ATUAL:",
            json.dumps({"id": m["origem"]["pergunta_id"], "materia": item["materia"],
                        "tema": item["tema"], "subtema": m["subtema"],
                        "dificuldade": item["dificuldade"], "pergunta": item["pergunta"],
                        "resposta": item["resposta"], "pontos_chave": m["pontos_chave"],
                        "fundamentos_legais": m["fundamentos_legais"]}, ensure_ascii=False), "",
            "APONTAMENTOS DO REVISOR:",
            json.dumps({"veredito": r.get("veredito"), "problemas": r.get("problemas", []),
                        "dispositivos_suspeitos": r.get("dispositivos_suspeitos", []),
                        "citacoes_nao_ancoradas": m["citacoes_nao_ancoradas"]},
                       ensure_ascii=False), "",
            "TRANSCRIÇÃO ORIGINAL (âncora do recorte, não fonte de correção):",
            json.dumps({"reacao_examinador": m["reacao_examinador"],
                        "pergunta_transcrita": m["origem"]["pergunta_transcrita"],
                        "resposta_transcrita": m["origem"]["resposta_transcrita"]},
                       ensure_ascii=False), "",
            "Retorne o JSON com 1 item em itens, com o mesmo id do ITEM ATUAL."])
        cid = "p3_" + hash_curto([VERSAO_PROMPT, MODELO_LLM,
                                  m["origem"]["pergunta_id"], item["resposta"], r])
        reqs_p3.append({"custom_id": cid,
                        "system": SYSTEM_CORRECAO.format(cargo=prova["cargo"],
                                                         banca=prova["banca"]),
                        "user": user, "ids": [m["origem"]["pergunta_id"]],
                        "_item": item, "_prova": prova})

    print(f"\np3 regeneração: {len(reqs_p3)} itens reprovados "
          f"({100*len(reqs_p3)/max(len(todos),1):.1f}%)")
    res_p3, falhas_p3 = executar_lote(reqs_p3, "p3")
    assert not falhas_p3, f"p3 falhou: {falhas_p3[:3]}"
    for r in reqs_p3:
        itens = res_p3.get(r["custom_id"], {}).get("itens", [])
        if not itens:
            continue
        aplicar_conteudo(r["_item"], itens[0], r["_prova"])
        r["_item"]["_meta"]["regenerado"] = True
        # o veredito guardado descreve o texto ANTERIOR
        r["_item"]["_meta"]["revisao_llm"]["obsoleta"] = True
else:
    print("\np3 regeneração: nenhum item reprovado")

## 10. Escrever a saída

Cinco artefatos, os mesmos que o `estruturar_banco.py` produzia:

| arquivo | para quê |
|---|---|
| `banco_estruturado.json` | **o consolidado**: é o que o simulador consome |
| `prova_*/q0001.json` + `_prova.json` | uma pasta por prova, com metadados e índice |
| `revisao_manual.csv` | fila de revisão humana, priorizada por gravidade |
| `auditoria_citacoes.json` | proveniência conferida de cada dispositivo |

In [ ]:
def soma(itens, chave):
    v = [i["_meta"]["auditoria_citacoes"].get(chave, 0) for i in itens]
    return sum(x if isinstance(x, int) else len(x) for x in v)


consolidado = []
for prova in provas:
    finais = por_prova[prova["prova_id"]]
    for it in finais:
        finalizar_item(it, len(finais))

    # árvore UF/concurso/prova: uma pasta plana não mostraria de que concurso é cada prova
    dir_prova = (DIR_SAIDA / prova["uf"]
                 / f"{sigla_banca(prova['banca'], prova['uf'])}_{slug(prova['cargo'])}"
                 / prova["prova_id"])
    dir_prova.mkdir(parents=True, exist_ok=True)
    for it in finais:
        (dir_prova / f"{it['id']}.json").write_text(
            json.dumps(it, ensure_ascii=False, indent=2), encoding="utf-8")

    meta = {
        "prova_id": prova["prova_id"], "concurso_id": prova["concurso_id"],
        "video_id": prova["video_id"], "uf": prova["uf"],
        "banca": prova["banca"], "cargo": prova["cargo"],
        "edicao": prova["edicao"], "ano": prova["ano"],
        "grupo_tematico": prova["grupo_tematico"], "colocacao": prova["colocacao"],
        "n_pares_origem": len(prova["itens"]), "n_questoes": len(finais),
        "n_descartadas": len(descartadas_prova[prova["prova_id"]]),
        "n_regeneradas": sum(1 for i in finais if i["_meta"]["regenerado"]),
        "n_revisao_necessaria": sum(1 for i in finais if i["_meta"]["revisao_necessaria"]),
        "citacoes": {
            "n_conferidas": soma(finais, "n_citacoes_conferidas"),
            "n_nao_ancoradas_resposta": soma(finais, "nao_ancoradas_resposta"),
            "n_nao_ancoradas_fundamentos": soma(finais, "nao_ancoradas_fundamentos"),
            "n_artigos_sem_norma": soma(finais, "artigos_sem_norma"),
            "n_origem_divergente": soma(finais, "origem_divergente"),
            "n_refs_nao_verificaveis": soma(finais, "refs_nao_verificaveis"),
            "n_fundamentos": sum(len(i["_meta"]["fundamentos_legais"]) for i in finais),
            "n_fundamentos_transcricao": sum(
                1 for i in finais for f in i["_meta"]["fundamentos_legais"]
                if f["origem_verificada"] == "transcricao"),
        },
        "duracao_s": round(prova["itens"][-1]["fim"] - prova["itens"][0]["inicio"], 1),
        "materias": sorted({i["materia"] for i in finais}),
        "questoes": [{"id": i["id"], "ordem": i["ordem"], "materia": i["materia"],
                      "tema": i["tema"], "dificuldade": i["dificuldade"],
                      "tipo": i["_meta"]["tipo"], "pergunta": i["pergunta"]}
                     for i in finais],
        "descartadas": descartadas_prova[prova["prova_id"]],
    }
    (dir_prova / "_prova.json").write_text(
        json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    consolidado.extend(finais)
    print(f"  {prova['prova_id']}: {len(finais)} questões, "
          f"{meta['n_regeneradas']} regeneradas, {meta['n_revisao_necessaria']} p/ revisão")

(DIR_SAIDA / "banco_estruturado.json").write_text(
    json.dumps(consolidado, ensure_ascii=False, indent=2), encoding="utf-8")

# um banco por concurso, a unidade que o simulador importa
por_concurso = defaultdict(list)
for i in consolidado:
    por_concurso[i["concurso_id"]].append(i)
dir_conc = DIR_SAIDA / "por_concurso"
dir_conc.mkdir(exist_ok=True)
indice = []
for cid, itens in sorted(por_concurso.items()):
    (dir_conc / f"{cid}.json").write_text(
        json.dumps(itens, ensure_ascii=False, indent=2), encoding="utf-8")
    limpas = sum(1 for i in itens if not i["_meta"]["revisao_necessaria"])
    indice.append({"concurso_id": cid, "uf": itens[0]["uf"], "banca": itens[0]["banca"],
                   "cargo": itens[0]["cargo"],
                   "provas": sorted({i["prova_id"] for i in itens}),
                   "n_provas": len({i["prova_id"] for i in itens}),
                   "n_questoes": len(itens), "n_sem_revisao": limpas,
                   "materias": dict(Counter(i["materia"] for i in itens).most_common())})
(DIR_SAIDA / "indice_concursos.json").write_text(
    json.dumps(sorted(indice, key=lambda x: (x["uf"], x["concurso_id"])),
               ensure_ascii=False, indent=2), encoding="utf-8")

print(f"\n{len(indice)} concursos em {len({x['uf'] for x in indice})} estados:")
for x in sorted(indice, key=lambda y: (y["uf"], -y["n_questoes"])):
    print(f"  {x['uf']}  {x['concurso_id']:26} {x['n_provas']:2} prova(s)  "
          f"{x['n_questoes']:4} questões  {x['n_sem_revisao']:4} sem revisão")

# fila de revisao, priorizada por gravidade
prio = {"regenerado_conferir": 0, "revisor_grave": 0, "citacao_nao_ancorada": 1,
        "dispositivo_suspeito": 1, "citacao_sem_norma": 2, "revisor_ajustes": 2,
        "confianca_baixa": 3}
revisao = sorted((i for i in consolidado if i["_meta"]["revisao_necessaria"]),
                 key=lambda i: min(prio.get(m, 5) for m in i["_meta"]["motivos_revisao"]))
with open(DIR_SAIDA / "revisao_manual.csv", "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f, delimiter=";")
    w.writerow(["uid", "prova_id", "id", "materia", "tema", "colocacao", "confianca",
                "gabarito_status", "regenerado", "motivos", "veredito_revisor",
                "problemas_revisor", "dispositivos_suspeitos", "citacoes_nao_ancoradas",
                "artigos_sem_norma", "origem_divergente", "pergunta", "resposta"])
    for i in revisao:
        m = i["_meta"]; r = m.get("revisao_llm") or {}; a = m["auditoria_citacoes"]
        pre = "[PRÉ-REGENERAÇÃO] " if r.get("obsoleta") else ""
        w.writerow([i["uid"], i["prova_id"], i["id"], i["materia"], i["tema"],
                    m["colocacao_candidato"], m["confianca"], m["gabarito_status"],
                    m["regenerado"], "|".join(m["motivos_revisao"]), r.get("veredito", ""),
                    pre + " | ".join(r.get("problemas", [])),
                    pre + " | ".join(r.get("dispositivos_suspeitos", [])),
                    " | ".join(m["citacoes_nao_ancoradas"]),
                    " | ".join(a.get("artigos_sem_norma", [])),
                    " | ".join(a.get("origem_divergente", [])),
                    i["pergunta"], i["resposta"]])

auditoria = {
    "n_questoes": len(consolidado),
    "n_citacoes_conferidas": soma(consolidado, "n_citacoes_conferidas"),
    "n_fundamentos": sum(len(i["_meta"]["fundamentos_legais"]) for i in consolidado),
    "n_origem_divergente": soma(consolidado, "origem_divergente"),
    "n_refs_nao_verificaveis": soma(consolidado, "refs_nao_verificaveis"),
    "itens": [{"uid": i["uid"], **i["_meta"]["auditoria_citacoes"]} for i in consolidado
              if any(i["_meta"]["auditoria_citacoes"].get(k) for k in
                     ("nao_ancoradas_resposta", "nao_ancoradas_fundamentos",
                      "artigos_sem_norma", "origem_divergente", "refs_nao_verificaveis"))],
}
(DIR_SAIDA / "auditoria_citacoes.json").write_text(
    json.dumps(auditoria, ensure_ascii=False, indent=2), encoding="utf-8")

print("\n" + "=" * 72)
print(f"questões finais: {len(consolidado)} / {len(banco)} pares de origem")
print(f"regeneradas    : {sum(1 for i in consolidado if i['_meta']['regenerado'])}")
print(f"revisão manual : {len(revisao)} ({100*len(revisao)/max(len(consolidado),1):.1f}%)")
print(f"citações       : {auditoria['n_citacoes_conferidas']} conferidas | "
      f"{auditoria['n_origem_divergente']} com origem declarada errada")
print(f"\nsaída em {DIR_SAIDA}")

## 11. Conferência final

O schema tem de bater com o que o simulador consome. Esta célula compara campo a
campo com um banco de referência, se houver um à mão, e roda as checagens que
pegam quase todo erro sistemático.

In [ ]:
TOPO = {"uid","id","prova_id","concurso_id","uf","banca","cargo","ordem","materia",
        "tema","dificuldade","momento","pergunta","resposta","embedding_text","_meta"}
META = {"subtema","tipo","pontos_chave","fundamentos_legais","gabarito_status",
        "confianca","citacoes_nao_ancoradas","auditoria_citacoes","colocacao_candidato",
        "reacao_examinador","observacao","revisao_llm","regenerado",
        "revisao_necessaria","motivos_revisao","origem"}

problemas = []
for i in consolidado:
    if set(i) != TOPO:
        problemas.append(("topo", i["uid"], set(i) ^ TOPO))
    extra = set(i["_meta"]) - META - {"obsoleta"}
    falta = META - set(i["_meta"])
    if extra or falta:
        problemas.append(("_meta", i["uid"], falta, extra))
    if i["materia"] not in MATERIAS:
        problemas.append(("materia", i["uid"], i["materia"]))
    if i["momento"] != momento(i["ordem"], sum(1 for x in consolidado
                                               if x["prova_id"] == i["prova_id"])):
        problemas.append(("momento", i["uid"]))
    if i["embedding_text"] != texto_embedding(i):
        problemas.append(("embedding_text", i["uid"]))

uids = [i["uid"] for i in consolidado]
print(f"schema uniforme        : {'OK' if not problemas else problemas[:3]}")
print(f"uid único              : {len(set(uids)) == len(uids)}")
for p in provas:
    sub = sorted((i for i in consolidado if i["prova_id"] == p["prova_id"]),
                 key=lambda x: x["ordem"])
    ok = [x["ordem"] for x in sub] == list(range(1, len(sub) + 1))
    if not ok:
        print(f"  ORDEM QUEBRADA em {p['prova_id']}")
print(f"ordem 1..N contígua    : OK")
print(f"conservação            : {len(consolidado)} + "
      f"{sum(len(v) for v in descartadas_prova.values())} = {len(banco)} -> "
      f"{len(consolidado) + sum(len(v) for v in descartadas_prova.values()) == len(banco)}")

print(f"\ndistribuições:")
print(f"  matéria     : {dict(Counter(i['materia'] for i in consolidado).most_common(5))}")
print(f"  dificuldade : {dict(Counter(i['dificuldade'] for i in consolidado))}")
print(f"  momento     : {dict(Counter(i['momento'] for i in consolidado))}")
print(f"  gab. status : {dict(Counter(i['_meta']['gabarito_status'] for i in consolidado))}")
conf = [i["_meta"]["confianca"] for i in consolidado]
if conf:
    print(f"  confiança   : min {min(conf)} | mediana {sorted(conf)[len(conf)//2]} | "
          f"média {sum(conf)/len(conf):.3f}")

# comparacao opcional com um banco de referencia
REFERENCIA = Path("../src/saida_sol_20260903/banco_estruturado.json")
if REFERENCIA.exists():
    ref = json.loads(REFERENCIA.read_text(encoding="utf-8"))[0]
    print(f"\ncomparação com {REFERENCIA.name}:")
    print(f"  campos do topo só na referência : {sorted(set(ref) - TOPO) or 'nenhum'}")
    print(f"  campos de _meta só na referência: {sorted(set(ref['_meta']) - META) or 'nenhum'}")
    print(f"  -> schema {'COMPATÍVEL' if not (set(ref) - TOPO) and not (set(ref['_meta']) - META) else 'DIVERGENTE'}")

## 12. Fechamento

`dados/saida/banco_estruturado.json` está pronto e no mesmo schema que o simulador
consome. Dois caminhos daqui:

**Selecionar as melhores.** `src/montar_banco_final.py` corta o banco preservando as
cadeias de arguição. Aponte-o para esta saída:

```bash
cd ../src && python montar_banco_final.py --entrada "../montagem de banco de perguntas/dados/saida"
```

**Carregar no simulador.** `scripts/importar_banco.py` importa e vetoriza.

### Antes de usar as questões

Leia `revisao_manual.csv`. A fila vem priorizada: `regenerado_conferir` e
`revisor_grave` primeiro, `confianca_baixa` por último. Em rodadas anteriores ~21%
das questões caíram nessa fila: é trabalho humano real, e é o que separa um banco
utilizável de um banco confiante e errado.

### Ao rodar em outro concurso

Três coisas mudam:

1. **`MATERIAS`**: a taxonomia do edital. Texto livre destrói a agregação.
2. **`banca` e `cargo` no CSV**: entram nos prompts das etapas 6 a 9.
3. **`PROMPT_JURIDICO`**: o vocabulário que enviesa o Whisper. Concurso de outra
   área precisa do vocabulário dela.

O resto (diarização, alinhamento, atribuição de papel, blocos, as três passadas e a
auditoria de citações) vale para qualquer prova oral com uma banca e um candidato.

### Custo e tempo, por vídeo de ~90 min

| etapa | ordem de grandeza |
|---|---|
| download | 1-3 min |
| transcrição (GPU) | 10-20 min |
| diarização (GPU) | 3-5 min |
| atribuição de papel | ~20 chamadas |
| pares | ~30 chamadas |
| estruturação + revisão + correção | ~30 chamadas |

A conta de API fica na casa de poucos dólares por vídeo com `gpt-4o`. Tudo é
cacheado: reexecutar não paga de novo.